In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
os.environ['GOOGLE_API_KEY'] = 'YOUR_API_KEY_HERE'

In [3]:
!pip install -q sentence-transformers faiss-cpu rouge-score tqdm rank-bm25 xgboost google-genai

In [4]:
!pip install -q sentence-transformers faiss-cpu rouge-score tqdm google-genai pylcs

In [ ]:
"""
=============================================================================
MBR SELECTION + THREE-COLUMN OPTIMIZATION  (v2 — fast, leak-safe, cached)
=============================================================================
  TargetR1F1 → MBR with ROUGE-1 consensus
  TargetRLF1 → MBR with ROUGE-L consensus
  TargetLLM  → Gemini refinement (fallback: top-1 retrieval)
Fixes vs v1:
  - Pairwise utilities computed ONCE per query -> grid search is instant
  - Split-half tuning (tune even / confirm odd) + auto-revert to top-1
  - Test-time keeps exact-duplicate training questions (free points)
  - Embeddings + candidates cached to Drive (safe to rerun)
  - Submission column order from SampleSubmission
=============================================================================
"""
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['USE_TF'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import numpy as np, pandas as pd, torch, faiss, gc, json, time, pickle
from pathlib import Path
from tqdm import tqdm
from collections import Counter
from datetime import datetime
from rouge_score import rouge_scorer
from rouge_score import tokenize as rs_tokenize

def log(msg): print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}", flush=True)

# fast LCS (C++ if available, pure-python fallback)
try:
    import pylcs
    HAVE_PYLCS = True
except ImportError:
    try:
        import subprocess
        subprocess.run(['pip', 'install', '-q', 'pylcs'], check=True)
        import pylcs
        HAVE_PYLCS = True
    except Exception:
        HAVE_PYLCS = False
log(f"pylcs available: {HAVE_PYLCS}")

# ============================================================
# PATHS + DATA
# ============================================================
try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    else:
        log("Google Drive already mounted.")
    BASE = Path('/content/drive/MyDrive/multilingual-health-qa')
    DATA_DIR, OUTPUT_DIR = BASE / 'data', BASE / 'outputs'
    AFRIE5_DIR = OUTPUT_DIR / 'afrie5-final-model'
except ImportError:
    DATA_DIR, OUTPUT_DIR, AFRIE5_DIR = Path('data/raw/'), Path('outputs/'), None
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE = OUTPUT_DIR / 'mbr_cache'; CACHE.mkdir(exist_ok=True)

log("Loading data...")
train_df = pd.read_csv(DATA_DIR / 'Train.csv')
val_df   = pd.read_csv(DATA_DIR / 'Val.csv')
test_df  = pd.read_csv(DATA_DIR / 'Test.csv')
sample_sub = pd.read_csv(DATA_DIR / 'SampleSubmission.csv')
SUB_COLS = list(sample_sub.columns)

combined = pd.concat([train_df, val_df], ignore_index=True).dropna(subset=['input', 'output']).reset_index(drop=True)
questions_raw = combined['input'].fillna('').astype(str).tolist()
answers_raw   = combined['output'].fillna('').astype(str).tolist()
subsets_raw   = combined['subset'].fillna('').astype(str).tolist()
corpus_q_stripped = [q.strip() for q in questions_raw]
log(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}, Combined: {len(combined)}")
if torch.cuda.is_available():
    log(f"GPU: {torch.cuda.get_device_name(0)} | {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

SUBSET_TO_LANG = {
    'Aka_Gha': 'Akan (Ghana)', 'Amh_Eth': 'Amharic (Ethiopia)',
    'Eng_Eth': 'English (Ethiopia)', 'Eng_Gha': 'English (Ghana)',
    'Eng_Ken': 'English (Kenya)', 'Eng_Uga': 'English (Uganda)',
    'Lug_Uga': 'Luganda (Uganda)', 'Swa_Ken': 'Swahili (Kenya)',
}

scorer_both = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=False)

# ============================================================
# STEP 0: AMHARIC TOKENIZATION CHECK
# ============================================================
log(f"\n{'='*60}\nSTEP 0: Amharic ROUGE tokenization check\n{'='*60}")
amh = "የጤና ጥያቄ ምን ይመስላል"
r = scorer_both.score(amh, amh)
log(f"Amharic self-ROUGE: {r['rouge1'].fmeasure:.4f}")
log("Ge'ez invisible to scorer -> Amharic ROUGE points come from Latin tokens only."
    if r['rouge1'].fmeasure < 0.5 else "Amharic tokenization works.")

# ============================================================
# STEP 1: LOAD MODEL + EMBEDDINGS (cached)
# ============================================================
log(f"\n{'='*60}\nSTEP 1: Embeddings (cached to Drive)\n{'='*60}")
PREFIX = "query: "
val_qs  = val_df['input'].fillna('').astype(str).tolist()
test_qs = test_df['input'].fillna('').astype(str).tolist()
test_subs = test_df['subset'].fillna('').astype(str).tolist()

emb_files = {k: CACHE / f'emb_{k}.npy' for k in ('corpus', 'val', 'test')}
if all(f.exists() for f in emb_files.values()):
    log("Loading cached embeddings from Drive...")
    corpus_emb = np.load(emb_files['corpus'])
    val_emb    = np.load(emb_files['val'])
    test_emb   = np.load(emb_files['test'])
    assert len(corpus_emb) == len(combined) and len(val_emb) == len(val_df) and len(test_emb) == len(test_df), \
        "Cache size mismatch — delete mbr_cache/ and rerun"
else:
    from sentence_transformers import SentenceTransformer
    bienc = SentenceTransformer(str(AFRIE5_DIR) if AFRIE5_DIR and AFRIE5_DIR.exists()
        else 'McGill-NLP/AfriE5-Large-instruct', device='cuda:0')
    log(f"AfriE5: {sum(p.numel() for p in bienc.parameters())/1e6:.0f}M params")
    enc = lambda texts: bienc.encode([f"{PREFIX}{t}" for t in texts], batch_size=64,
        show_progress_bar=True, normalize_embeddings=True).astype(np.float32)
    log("Encoding corpus..."); corpus_emb = enc(questions_raw); np.save(emb_files['corpus'], corpus_emb)
    log("Encoding val...");    val_emb    = enc(val_qs);        np.save(emb_files['val'], val_emb)
    log("Encoding test...");   test_emb   = enc(test_qs);       np.save(emb_files['test'], test_emb)
    bienc.cpu(); del bienc; gc.collect(); torch.cuda.empty_cache()
    log("GPU freed. Embeddings cached.")

# Per-language FAISS indices
lang_indices = {}
for sub in sorted(set(subsets_raw)):
    mask = [i for i, s in enumerate(subsets_raw) if s == sub]
    idx = faiss.IndexFlatIP(corpus_emb.shape[1]); idx.add(corpus_emb[mask])
    lang_indices[sub] = (idx, mask)
    log(f"  {sub}: {len(mask)} samples")
global_idx = faiss.IndexFlatIP(corpus_emb.shape[1]); global_idx.add(corpus_emb)

# ============================================================
# STEP 2: CANDIDATES + FAST MBR MACHINERY
# ============================================================
K_CANDIDATES = 15
MAX_UTIL_TOKENS = 80   # truncation for consensus computation ONLY (ref scoring is full-length)

def get_same_lang_candidates(q_text, q_emb, subset, k=K_CANDIDATES, exclude_exact=True):
    """Top-k same-language candidates. exclude_exact=True for val (self-leak guard),
       False at test time (exact train duplicates are FREE POINTS)."""
    q_stripped = q_text.strip()
    if subset in lang_indices:
        idx, mask = lang_indices[subset]
        D, I = idx.search(q_emb.reshape(1, -1), min(k + 5, len(mask)))
        out = []
        for d, li in zip(D[0], I[0]):
            if li < 0: continue
            ci = mask[int(li)]
            if exclude_exact and corpus_q_stripped[ci] == q_stripped: continue
            out.append({'answer': answers_raw[ci], 'sim': float(d), 'idx': ci})
            if len(out) >= k: break
        if out: return out
    D, I = global_idx.search(q_emb.reshape(1, -1), k + 5)
    out = []
    for d, ci in zip(D[0], I[0]):
        ci = int(ci)
        if ci < 0: continue
        if exclude_exact and corpus_q_stripped[ci] == q_stripped: continue
        out.append({'answer': answers_raw[ci], 'sim': float(d), 'idx': ci})
        if len(out) >= k: break
    return out

def _toks(t): return rs_tokenize.tokenize(t, None)[:MAX_UTIL_TOKENS]

def _lcs_py(a, b):
    if not a or not b: return 0
    dp = [0]*(len(b)+1)
    for ai in a:
        prev = 0
        for j, bj in enumerate(b):
            cur = dp[j+1]
            dp[j+1] = prev+1 if ai == bj else (dp[j+1] if dp[j+1] >= dp[j] else dp[j])
            prev = cur
    return dp[-1]

def _lcs(a, b, vocab):
    if HAVE_PYLCS:
        sa = ''.join(chr(0x4E00 + vocab[t]) for t in a)
        sb = ''.join(chr(0x4E00 + vocab[t]) for t in b)
        return pylcs.lcs_sequence_length(sa, sb)
    return _lcs_py(a, b)

def prep_query(cands):
    """Once per query. Returns (dedup_answers, weights, util1_base, utilL_base) —
       MBR at any (alpha, margin) is then pure arithmetic."""
    answers = [c['answer'] for c in cands]
    w = np.exp(np.array([c['sim'] for c in cands]) * 5); w /= w.sum()
    seen, dd, ddw = {}, [], []
    for a, wi in zip(answers, w):
        key = a.strip().lower()
        if key in seen: ddw[seen[key]] += wi
        else: seen[key] = len(dd); dd.append(a); ddw.append(wi)
    ddw = np.array(ddw); ddw /= ddw.sum()
    n = len(dd)
    toks = [_toks(a) for a in dd]
    if n == 1:
        return dd, ddw, np.zeros(1), np.zeros(1)
    vocab = {}
    for t in toks:
        for tok in t:
            if tok not in vocab: vocab[tok] = len(vocab)
    cnts = [Counter(t) for t in toks]; lens = [len(t) for t in toks]
    S1, SL = np.zeros((n, n)), np.zeros((n, n))
    for i in range(n):
        for j in range(i+1, n):
            if lens[i] and lens[j]:
                ov = sum((cnts[i] & cnts[j]).values())
                S1[i, j] = S1[j, i] = 2*ov/(lens[i]+lens[j])
                l = _lcs(toks[i], toks[j], vocab)
                SL[i, j] = SL[j, i] = 2*l/(lens[i]+lens[j])
    return dd, ddw, S1 @ ddw, SL @ ddw

def mbr_idx(util_base, ddw, alpha, margin):
    """Index of chosen candidate. margin=99 -> never override -> top-1."""
    if len(ddw) == 1: return 0
    util = util_base + alpha * ddw
    b = int(np.argmax(util))
    return b if (b != 0 and util[b] - util[0] > margin) else 0

# ============================================================
# STEP 3: VAL PRECOMPUTE + GRID SEARCH (split-half, auto-revert)
# ============================================================
log(f"\n{'='*60}\nSTEP 3: Val precompute + per-language tuning\n{'='*60}")

cands_cache = CACHE / 'val_cands.pkl'
if cands_cache.exists():
    val_cands_all = pickle.load(open(cands_cache, 'rb'))
    log(f"Loaded cached val candidates ({len(val_cands_all)})")
else:
    val_cands_all = [get_same_lang_candidates(val_qs[i].strip(), val_emb[i],
                     str(val_df.iloc[i]['subset']), exclude_exact=True)
                     for i in tqdm(range(len(val_df)), desc="Val candidates")]
    pickle.dump(val_cands_all, open(cands_cache, 'wb'))

prep_cache = CACHE / 'val_prep.pkl'
if prep_cache.exists():
    val_prep, val_refscores = pickle.load(open(prep_cache, 'rb'))
    log("Loaded cached val prep + ref scores")
else:
    val_prep, val_refscores = [], []
    for i in tqdm(range(len(val_df)), desc="Pairwise utils + ref scores"):
        cands = val_cands_all[i]
        ref = str(val_df.iloc[i]['output']).strip()
        if not cands or not ref:
            val_prep.append(None); val_refscores.append(None); continue
        dd, ddw, u1, uL = prep_query(cands)
        val_prep.append((dd, ddw, u1, uL))
        rs = [scorer_both.score(ref, c) for c in dd]   # FULL-length scoring vs reference
        val_refscores.append([(x['rouge1'].fmeasure, x['rougeL'].fmeasure) for x in rs])
    pickle.dump((val_prep, val_refscores), open(prep_cache, 'wb'))

alphas_to_test  = [0.05, 0.10, 0.15, 0.20]
margins_to_test = [0.0, 0.005, 0.01, 0.02, 0.03, 0.05, 0.10]
TOP1 = (0.15, 99.0)   # never override

def combo_score(idx_list, alpha, margin):
    r1 = [val_refscores[i][mbr_idx(val_prep[i][2], val_prep[i][1], alpha, margin)][0] for i in idx_list]
    rl = [val_refscores[i][mbr_idx(val_prep[i][3], val_prep[i][1], alpha, margin)][1] for i in idx_list]
    return np.mean(r1) + np.mean(rl)

best_params = {}
log(f"\n{'Sub':<10} {'α':>5} {'τ':>6} {'holdout Δ':>10}")
for sub in sorted(SUBSET_TO_LANG.keys()):
    idxs = [i for i in range(len(val_df))
            if str(val_df.iloc[i]['subset']) == sub and val_prep[i] is not None]
    if not idxs: continue
    tune = idxs[0::2]; hold = idxs[1::2]
    best = max(((a, m) for a in alphas_to_test for m in margins_to_test),
               key=lambda p: combo_score(tune, *p))
    gain = combo_score(hold, *best) - combo_score(hold, *TOP1)
    if gain <= 0:
        best = TOP1   # doesn't generalize -> stay top-1 for this language
    best_params[sub] = best
    log(f"{sub:<10} {best[0]:>5.2f} {best[1]:>6.3f} {gain:>+10.4f}{'  ★' if gain > 0.003 else ''}")

# ============================================================
# RESULTS: full-val per-language table
# ============================================================
log(f"\n{'='*60}\nRESULTS: MBR vs top-1 baseline (full val)\n{'='*60}")
log(f"{'Sub':<10} {'Base R1':>8} {'MBR R1':>8} {'Δ R1':>8} {'Base RL':>8} {'MBR RL':>8} {'Δ RL':>8}")
log('-'*62)
tot = {k: [] for k in ('br1','brl','mr1','mrl')}
for sub in sorted(SUBSET_TO_LANG.keys()):
    idxs = [i for i in range(len(val_df))
            if str(val_df.iloc[i]['subset']) == sub and val_prep[i] is not None]
    if not idxs: continue
    a, m = best_params[sub]
    br1 = [val_refscores[i][0][0] for i in idxs]
    brl = [val_refscores[i][0][1] for i in idxs]
    mr1 = [val_refscores[i][mbr_idx(val_prep[i][2], val_prep[i][1], a, m)][0] for i in idxs]
    mrl = [val_refscores[i][mbr_idx(val_prep[i][3], val_prep[i][1], a, m)][1] for i in idxs]
    for k, v in zip(('br1','brl','mr1','mrl'), (br1, brl, mr1, mrl)): tot[k] += v
    log(f"{sub:<10} {np.mean(br1):>8.4f} {np.mean(mr1):>8.4f} {np.mean(mr1)-np.mean(br1):>+8.4f} "
        f"{np.mean(brl):>8.4f} {np.mean(mrl):>8.4f} {np.mean(mrl)-np.mean(brl):>+8.4f}")
ov = {k: np.mean(v) for k, v in tot.items()}
log('-'*62)
log(f"{'OVERALL':<10} {ov['br1']:>8.4f} {ov['mr1']:>8.4f} {ov['mr1']-ov['br1']:>+8.4f} "
    f"{ov['brl']:>8.4f} {ov['mrl']:>8.4f} {ov['mrl']-ov['brl']:>+8.4f}")
est_lb = (ov['mr1'] + ov['mrl'] + 0.775) / 3
log(f"\nEstimated LB (MBR ROUGE + retrieval LLM): {est_lb:.4f}   (prev best: 0.6545)")

# ============================================================
# STEP 4: TEST SUBMISSIONS  (exclude_exact=False — keep duplicates!)
# ============================================================
log(f"\n{'='*60}\nSTEP 4: Test submissions\n{'='*60}")
rows_mbr = []
test_cands_cache = {}
for i in tqdm(range(len(test_df)), desc="MBR test"):
    rid = test_df.iloc[i]['ID']
    cands = get_same_lang_candidates(test_qs[i].strip(), test_emb[i], test_subs[i],
                                     exclude_exact=False)
    test_cands_cache[str(rid)] = cands
    if not cands:
        rows_mbr.append({'ID': rid, 'TargetR1F1': 'No answer',
                         'TargetRLF1': 'No answer', 'TargetLLM': 'No answer'})
        continue
    dd, ddw, u1, uL = prep_query(cands)
    a, m = best_params.get(test_subs[i], (0.15, 0.02))
    rows_mbr.append({'ID': rid,
                     'TargetR1F1': dd[mbr_idx(u1, ddw, a, m)],
                     'TargetRLF1': dd[mbr_idx(uL, ddw, a, m)],
                     'TargetLLM':  cands[0]['answer']})

sub_mbr = pd.DataFrame(rows_mbr).reindex(columns=SUB_COLS)
assert len(sub_mbr) == len(sample_sub) and not sub_mbr.isnull().any().any()
sub_mbr.to_csv(OUTPUT_DIR / 'submission_mbr.csv', index=False)
log("Saved: submission_mbr.csv  (MBR R1 + MBR RL + top-1 LLM)")

# ============================================================
# STEP 5: GEMINI LLM COLUMN (resumable)
# ============================================================
log(f"\n{'='*60}\nSTEP 5: Gemini LLM column\n{'='*60}")
api_key = os.environ.get('YOUR_API_KEY_HERE') or os.environ.get('GOOGLE_API_KEY')
if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get('GEMINI_API_KEY') or userdata.get('GOOGLE_API_KEY')
    except Exception: pass

GEMINI_OK = False
if api_key:
    try:
        from google import genai
        client = genai.Client(api_key=api_key)
        t = client.models.generate_content(model='gemini-2.0-flash', contents='Say OK',
            config=genai.types.GenerateContentConfig(temperature=0, max_output_tokens=5))
        log(f"Gemini API test: {t.text.strip()}"); GEMINI_OK = True
    except Exception as e:
        log(f"Gemini API failed: {e}")
else:
    log("No Gemini key found — skipping LLM column (submission_mbr.csv still valid).")

if GEMINI_OK:
    call_count, call_start = 0, time.time()
    def gemini_call(prompt, temp=0.3, max_tok=600, retries=3):
        global call_count, call_start
        for attempt in range(retries):
            try:
                call_count += 1
                el = time.time() - call_start
                if el < 60 and call_count > 14:
                    time.sleep(61 - el); call_count, call_start = 0, time.time()
                resp = client.models.generate_content(model='gemini-2.0-flash', contents=prompt,
                    config=genai.types.GenerateContentConfig(temperature=temp, max_output_tokens=max_tok))
                return resp.text.strip()
            except Exception as e:
                if '429' in str(e).lower() or 'quota' in str(e).lower():
                    w = min(30*(attempt+1), 120); log(f"  Rate limited, waiting {w}s...")
                    time.sleep(w); call_count, call_start = 0, time.time()
                elif attempt < retries - 1: time.sleep(2 ** attempt)
                else: return None
        return None

    llm_prog = OUTPUT_DIR / 'gemini_mbr_llm_prog.json'
    llm_ans = json.load(open(llm_prog)) if llm_prog.exists() else {}
    log(f"Resume: {len(llm_ans)} answers done")
    for i in tqdm(range(len(test_df)), desc="Gemini LLM"):
        rid = str(test_df.iloc[i]['ID'])
        if rid in llm_ans: continue
        cands = test_cands_cache.get(rid, [])[:5]
        if not cands:
            llm_ans[rid] = "No answer."; continue
        lang = SUBSET_TO_LANG.get(test_subs[i], test_subs[i])
        ctx = "\n".join(f"{k+1}. {c['answer']}" for k, c in enumerate(cands))
        prompt = (f"You are a professional multilingual health expert. Answer this health question in {lang}.\n"
                  f"Question: {test_qs[i]}\nReference answers for context:\n{ctx}\n"
                  f"Base your answer strictly on the references above. Merge complementary points. "
                  f"Be complete, medically accurate, and direct. Do not mention the references. "
                  f"Answer in {lang}:")
        gen = gemini_call(prompt)
        llm_ans[rid] = gen if gen else cands[0]['answer']
        if (i+1) % 100 == 0:
            json.dump(llm_ans, open(llm_prog, 'w'))
    json.dump(llm_ans, open(llm_prog, 'w'))
    log(f"Gemini complete: {len(llm_ans)} answers")

    rows_gem = [{**row, 'TargetLLM': llm_ans.get(str(row['ID']), row['TargetLLM'])}
                for row in rows_mbr]
    sub_gem = pd.DataFrame(rows_gem).reindex(columns=SUB_COLS)
    sub_gem.to_csv(OUTPUT_DIR / 'submission_mbr_gemini.csv', index=False)
    log("Saved: submission_mbr_gemini.csv  (MBR ROUGE + Gemini LLM)")

# ============================================================
# FINAL SUMMARY
# ============================================================
log(f"\n{'='*60}\nDONE\n{'='*60}")
log(f"Val:  top-1  R1={ov['br1']:.4f} RL={ov['brl']:.4f}")
log(f"Val:  MBR    R1={ov['mr1']:.4f} RL={ov['mrl']:.4f}  "
    f"(Δ R1 {ov['mr1']-ov['br1']:+.4f}, Δ RL {ov['mrl']-ov['brl']:+.4f})")
log(f"Params: " + ", ".join(f"{s}(α={a},τ={m})" for s, (a, m) in sorted(best_params.items())))
log(f"\nEstimated LB (MBR + retrieval LLM): {est_lb:.4f}")
if GEMINI_OK:
    log(f"Estimated LB (MBR + Gemini @0.85):  {(ov['mr1']+ov['mrl']+0.85)/3:.4f}")
log("\nSubmit order: 1) submission_mbr.csv (isolates ROUGE gain)  2) submission_mbr_gemini.csv")

[06:38:12] pylcs available: True
[06:38:12] Google Drive already mounted.
[06:38:12] Loading data...
[06:38:15] Train: 29815, Val: 6686, Test: 2618, Combined: 36501
[06:38:16] GPU: Tesla T4 | 15.6 GB
[06:38:16] 
STEP 0: Amharic ROUGE tokenization check
[06:38:16] Amharic self-ROUGE: 0.0000
[06:38:16] Ge'ez invisible to scorer -> Amharic ROUGE points come from Latin tokens only.
[06:38:16] 
STEP 1: Embeddings (cached to Drive)
[06:38:16] Loading cached embeddings from Drive...
[06:38:24]   Aka_Gha: 5569 samples
[06:38:24]   Amh_Eth: 2307 samples
[06:38:24]   Eng_Eth: 4479 samples
[06:38:24]   Eng_Gha: 5547 samples
[06:38:24]   Eng_Ken: 2470 samples
[06:38:24]   Eng_Uga: 9312 samples
[06:38:24]   Lug_Uga: 4229 samples
[06:38:24]   Swa_Ken: 2588 samples
[06:38:25] 
STEP 3: Val precompute + per-language tuning
[06:38:25] Loaded cached val candidates (6686)
[06:38:26] Loaded cached val prep + ref scores
[06:38:26] 
Sub            α      τ  holdout Δ
[06:38:27] Aka_Gha     0.20  0.030    +0.

MBR test: 100%|██████████| 2618/2618 [00:17<00:00, 145.60it/s]


[06:38:51] Saved: submission_mbr.csv  (MBR R1 + MBR RL + top-1 LLM)
[06:38:51] 
STEP 5: Gemini LLM column
[06:38:56] Gemini API failed: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.0-flash is no longer available. Please update your code to use a newer model for the latest features and improvements.', 'status': 'NOT_FOUND'}}
[06:38:56] 
DONE
[06:38:56] Val:  top-1  R1=0.6319 RL=0.5751
[06:38:56] Val:  MBR    R1=0.6397 RL=0.5788  (Δ R1 +0.0078, Δ RL +0.0037)
[06:38:56] Params: Aka_Gha(α=0.2,τ=0.03), Amh_Eth(α=0.05,τ=0.03), Eng_Eth(α=0.15,τ=99.0), Eng_Gha(α=0.2,τ=0.05), Eng_Ken(α=0.15,τ=99.0), Eng_Uga(α=0.15,τ=99.0), Lug_Uga(α=0.15,τ=99.0), Swa_Ken(α=0.15,τ=99.0)
[06:38:56] 
Estimated LB (MBR + retrieval LLM): 0.6645
[06:38:56] 
Submit order: 1) submission_mbr.csv (isolates ROUGE gain)  2) submission_mbr_gemini.csv


In [ ]:
# =============================================================================
# GEMINI LLM COLUMN — v3 (gemini-2.5-flash, thinking off, parallel, resumable)
# Produces: submission_mbr_gemini.csv + val sample for per-language gating
# =============================================================================
import os, json, time, threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from tqdm import tqdm
import numpy as np
import pandas as pd
from google import genai

# ---- API setup ----
api_key = os.environ.get('GOOGLE_API_KEY') or os.environ.get('GEMINI_API_KEY')
assert api_key, "Set GOOGLE_API_KEY first"
client = genai.Client(api_key=api_key)
MODEL = 'gemini-2.5-flash'

# ---- sanity test before burning calls ----
t = client.models.generate_content(
    model=MODEL, contents='Say OK',
    config=genai.types.GenerateContentConfig(
        temperature=0, max_output_tokens=50,
        thinking_config=genai.types.ThinkingConfig(thinking_budget=0)))
print(f"API test [{MODEL}]: {t.text.strip()}")

# ---- call wrapper: thinking OFF, loud failures ----
def gemini_call(prompt, temp=0.3, max_tok=600, retries=4):
    last_err = None
    for attempt in range(retries):
        try:
            r = client.models.generate_content(
                model=MODEL, contents=prompt,
                config=genai.types.GenerateContentConfig(
                    temperature=temp, max_output_tokens=max_tok,
                    thinking_config=genai.types.ThinkingConfig(thinking_budget=0)))
            txt = (r.text or "").strip()
            if txt:
                return txt
            last_err = "empty response"
        except Exception as e:
            last_err = str(e)[:150]
            if '429' in last_err or 'quota' in last_err.lower():
                time.sleep(min(20 * (attempt + 1), 60))
            else:
                time.sleep(2 ** attempt)
    print(f"  FAIL after {retries} tries: {last_err}")
    return None

def make_prompt(q, lang, cands):
    ctx = "\n".join(f"{k+1}. {c['answer']}" for k, c in enumerate(cands[:5]))
    return (f"You are a professional multilingual health expert. Answer this health question in {lang}.\n"
            f"Question: {q}\n"
            f"Reference answers for context:\n{ctx}\n"
            f"Base your answer strictly on the references above. Merge complementary points. "
            f"Be complete, medically accurate, and direct. Do not mention the references. "
            f"Answer in {lang}:")

# =============================================================================
# PART 1: TEST LLM COLUMN (parallel, resumes from progress file)
# =============================================================================
llm_prog = OUTPUT_DIR / 'gemini_mbr_llm_prog.json'
llm_ans = json.load(open(llm_prog)) if llm_prog.exists() else {}
lock = threading.Lock()
print(f"\nTest LLM resume: {len(llm_ans)}/{len(test_df)} already done")

def work_test(i):
    rid = str(test_df.iloc[i]['ID'])
    cands = test_cands_cache.get(rid, [])
    if not cands:
        return rid, "No answer."
    gen = gemini_call(make_prompt(
        test_qs[i], SUBSET_TO_LANG.get(test_subs[i], test_subs[i]), cands))
    return rid, (gen if gen else cands[0]['answer'])

todo = [i for i in range(len(test_df)) if str(test_df.iloc[i]['ID']) not in llm_ans]
if todo:
    with ThreadPoolExecutor(max_workers=20) as ex:
        futures = [ex.submit(work_test, i) for i in todo]
        for n, f in enumerate(tqdm(as_completed(futures), total=len(todo), desc="Gemini test")):
            rid, ans = f.result()
            with lock:
                llm_ans[rid] = ans
                if (n + 1) % 100 == 0:
                    json.dump(llm_ans, open(llm_prog, 'w'))
    json.dump(llm_ans, open(llm_prog, 'w'))
print(f"Test LLM complete: {len(llm_ans)}/{len(test_df)}")

# fallback count sanity check (how many ended up as retrieval fallback "No answer.")
n_noans = sum(1 for v in llm_ans.values() if v == "No answer.")
print(f"  'No answer.' fallbacks: {n_noans}")

# =============================================================================
# PART 2: BUILD SUBMISSION (MBR ROUGE columns + Gemini LLM column)
# =============================================================================
rows_gem = [{**row, 'TargetLLM': llm_ans.get(str(row['ID']), row['TargetLLM'])}
            for row in rows_mbr]
sub_gem = pd.DataFrame(rows_gem).reindex(columns=SUB_COLS)
assert len(sub_gem) == len(test_df) and not sub_gem.isnull().any().any()
sub_gem.to_csv(OUTPUT_DIR / 'submission_mbr_gemini.csv', index=False)
print("\nSaved: submission_mbr_gemini.csv  (MBR R1 + MBR RL + Gemini LLM)")

# =============================================================================
# PART 3: VAL SAMPLE (50/language) — for per-language Gemini-vs-retrieval gating
# =============================================================================
rng = np.random.default_rng(42)
val_sample = []
for sub in SUBSET_TO_LANG:
    idxs = [i for i in range(len(val_df))
            if str(val_df.iloc[i]['subset']) == sub and val_cands_all[i]]
    val_sample += list(rng.choice(idxs, min(50, len(idxs)), replace=False))
val_sample = [int(i) for i in val_sample]

vprog = OUTPUT_DIR / 'gemini_val_sample.json'
vans = json.load(open(vprog)) if vprog.exists() else {}
print(f"\nVal sample resume: {len(vans)}/{len(val_sample)}")

def work_val(i):
    gen = gemini_call(make_prompt(
        val_qs[i], SUBSET_TO_LANG.get(str(val_df.iloc[i]['subset'])), val_cands_all[i]))
    return str(i), (gen if gen else val_cands_all[i][0]['answer'])

vtodo = [i for i in val_sample if str(i) not in vans]
if vtodo:
    with ThreadPoolExecutor(max_workers=20) as ex:
        futures = [ex.submit(work_val, i) for i in vtodo]
        for n, f in enumerate(tqdm(as_completed(futures), total=len(vtodo), desc="Gemini val")):
            k, ans = f.result()
            with lock:
                vans[k] = ans
                if (n + 1) % 50 == 0:
                    json.dump(vans, open(vprog, 'w'))
    json.dump(vans, open(vprog, 'w'))
print(f"Val sample complete: {len(vans)}/{len(val_sample)}")

print("\n" + "=" * 60)
print("DONE. Submit order:")
print("  1. submission_mbr.csv         (if not yet submitted)")
print("  2. submission_mbr_gemini.csv  (adds Gemini LLM column)")
print("=" * 60)

API test [gemini-2.5-flash]: OK

Test LLM resume: 2618/2618 already done
Test LLM complete: 2618/2618
  'No answer.' fallbacks: 0

Saved: submission_mbr_gemini.csv  (MBR R1 + MBR RL + Gemini LLM)

Val sample resume: 400/400
Val sample complete: 400/400

DONE. Submit order:
  1. submission_mbr.csv         (if not yet submitted)
  2. submission_mbr_gemini.csv  (adds Gemini LLM column)


In [ ]:
# ==== JUDGE PROXY: top-1 vs MBR vs Gemini, per language ====
import re
def judge_prompt(q, ref, cand):
    return (f"You are grading an answer to a health question.\n"
            f"Question: {q}\n\nReference (gold) answer:\n{ref}\n\n"
            f"Candidate answer:\n{cand}\n\n"
            f"Score the candidate from 0 to 10 for factual agreement with the reference, "
            f"completeness, and relevance. Reply with ONLY the number.")

def get_score(q, ref, cand):
    r = gemini_call(judge_prompt(q, ref, cand), temp=0.0, max_tok=10)
    if not r: return None
    m = re.search(r'\d+\.?\d*', r)
    return min(float(m.group()), 10.0) if m else None

vans = json.load(open(OUTPUT_DIR / 'gemini_val_sample.json'))
jobs = []
for k in vans:
    i = int(k)
    ref = str(val_df.iloc[i]['output']).strip()
    if not ref or not val_prep[i]: continue
    dd, ddw, u1, uL = val_prep[i]
    a, m = best_params.get(str(val_df.iloc[i]['subset']), (0.15, 99.0))
    opts = {'top1': val_cands_all[i][0]['answer'],
            'mbr':  dd[mbr_idx(u1, ddw, a, m)],
            'gem':  vans[k]}
    for name, cand in opts.items():
        jobs.append((i, name, val_qs[i], ref, cand))

results = {}
with ThreadPoolExecutor(max_workers=20) as ex:
    futs = {ex.submit(get_score, q, r, c): (i, name) for i, name, q, r, c in jobs}
    for f in tqdm(as_completed(futs), total=len(futs), desc="Judging"):
        i, name = futs[f]
        s = f.result()
        if s is not None: results.setdefault(i, {})[name] = s

print(f"\n{'Sub':<10} {'n':>4} {'top1':>6} {'mbr':>6} {'gem':>6}  winner")
gate = {}
for sub in sorted(SUBSET_TO_LANG):
    rows = [results[i] for i in results
            if str(val_df.iloc[i]['subset']) == sub and len(results[i]) == 3]
    if not rows: continue
    means = {k: np.mean([r[k] for r in rows]) for k in ('top1', 'mbr', 'gem')}
    win = max(means, key=means.get)
    gate[sub] = win
    print(f"{sub:<10} {len(rows):>4} {means['top1']:>6.2f} {means['mbr']:>6.2f} "
          f"{means['gem']:>6.2f}  {win}")
print("\nGate:", gate)

Judging: 100%|██████████| 1200/1200 [01:39<00:00, 12.10it/s]


Sub           n   top1    mbr    gem  winner
Aka_Gha      50   4.22   3.92   6.22  gem
Amh_Eth      50   6.24   5.28   7.70  gem
Eng_Eth      50   7.70   7.66   8.54  gem
Eng_Gha      50   4.76   4.42   7.90  gem
Eng_Ken      50   9.10   9.10   9.36  gem
Eng_Uga      50   8.98   8.98   9.68  gem
Lug_Uga      50   8.54   8.54   8.62  gem
Swa_Ken      50   9.42   9.44   8.58  mbr

Gate: {'Aka_Gha': 'gem', 'Amh_Eth': 'gem', 'Eng_Eth': 'gem', 'Eng_Gha': 'gem', 'Eng_Ken': 'gem', 'Eng_Uga': 'gem', 'Lug_Uga': 'gem', 'Swa_Ken': 'mbr'}


In [ ]:
# =============================================================================
# EXTRACTIVE STITCHER — exceed the single-candidate oracle on weak languages
# Greedy sentence selection maximizing EXPECTED ROUGE-1 F1 against the
# candidate-consensus token distribution. Verbatim sentences only.
# Gated per language per column on val (split-half). Builds final submission
# with gated LLM column (Gemini everywhere except Swa_Ken -> top-1).
# =============================================================================
import re, pickle
from collections import Counter

STITCH_LANGS = ['Aka_Gha', 'Eng_Gha', 'Eng_Eth', 'Lug_Uga', 'Amh_Eth']  # skip near-ceiling langs
K_DEEP = 30          # deeper pool: oracle ranks 13-17 on Gha subsets need it
MAX_POOL_SENTS = 120
LAMBDAS = [0.7, 0.85, 1.0, 1.15]   # ref-length prior multiplier (tuned per language)

def full_toks(t): return rs_tokenize.tokenize(t, None)[:400]

# Per-language reference-length prior from the corpus (test refs come from same distribution)
ref_len_prior = {}
for sub in SUBSET_TO_LANG:
    lens = [len(full_toks(answers_raw[i])) for i, s in enumerate(subsets_raw) if s == sub]
    ref_len_prior[sub] = float(np.median(lens))
print("Ref-length priors:", {k: int(v) for k, v in ref_len_prior.items()})

def stitch(cands, lam, sub):
    """Greedy expected-R1-F1 sentence stitching. Returns stitched answer string."""
    answers = [c['answer'] for c in cands]
    w = np.exp(np.array([c['sim'] for c in cands]) * 5); w /= w.sum()
    # expected reference token counts from weighted candidate consensus
    p = Counter()
    for a, wi in zip(answers, w):
        for t, cnt in Counter(full_toks(a)).items():
            p[t] += wi * cnt
    ref_len = lam * ref_len_prior[sub]
    # verbatim sentence pool (deduped)
    seen, pool = set(), []
    for a in answers:
        for s in re.split(r'(?<=[.!?])\s+|\n+', a):
            s = s.strip()
            st = full_toks(s)
            if len(st) < 4: continue
            key = ' '.join(st)
            if key not in seen:
                seen.add(key); pool.append((s, Counter(st), len(st)))
            if len(pool) >= MAX_POOL_SENTS: break
        if len(pool) >= MAX_POOL_SENTS: break
    if not pool: return answers[0]

    def exp_f1(cov, hyp_len):
        m = sum(min(cnt, p[t]) for t, cnt in cov.items())
        if hyp_len == 0 or m == 0: return 0.0
        prec, rec = m / hyp_len, m / ref_len
        return 2 * prec * rec / (prec + rec)

    chosen, cov, hyp_len, cur = [], Counter(), 0, 0.0
    while pool:
        best_i, best_gain = -1, 0.0
        for i, (s, sc, sl) in enumerate(pool):
            nc = cov.copy(); nc.update(sc)
            g = exp_f1(nc, hyp_len + sl) - cur
            if g > best_gain: best_gain, best_i = g, i
        if best_i < 0: break
        s, sc, sl = pool.pop(best_i)
        chosen.append(s); cov.update(sc); hyp_len += sl; cur += best_gain
    return ' '.join(chosen) if chosen else answers[0]

# ---- deep val candidates for stitch languages (cached) ----
deep_cache = CACHE / 'val_cands_deep.pkl'
if deep_cache.exists():
    val_deep = pickle.load(open(deep_cache, 'rb'))
else:
    val_deep = {}
    for i in tqdm(range(len(val_df)), desc="Deep val candidates"):
        sub = str(val_df.iloc[i]['subset'])
        if sub in STITCH_LANGS:
            val_deep[i] = get_same_lang_candidates(val_qs[i].strip(), val_emb[i], sub,
                                                   k=K_DEEP, exclude_exact=True)
    pickle.dump(val_deep, open(deep_cache, 'wb'))
print(f"Deep val candidates: {len(val_deep)} queries")

# ---- evaluate + tune lambda per language (split-half, gate vs MBR per column) ----
print(f"\n{'Sub':<10} {'λ':>5} {'MBR R1':>8} {'Sti R1':>8} {'Δ R1':>8} "
      f"{'MBR RL':>8} {'Sti RL':>8} {'Δ RL':>8}  gates")
stitch_gate = {}   # sub -> {'r1': bool, 'rl': bool, 'lam': float}
stitch_val_text = {}  # (i, lam) cache of stitched strings to avoid recompute
for sub in STITCH_LANGS:
    idxs = [i for i in val_deep if val_prep[i] is not None
            and str(val_df.iloc[i]['output']).strip()]
    idxs = [i for i in idxs if str(val_df.iloc[i]['subset']) == sub]
    if not idxs: continue
    tune, hold = idxs[0::2], idxs[1::2]
    a_p, m_p = best_params.get(sub, (0.15, 99.0))

    def mbr_scores(ix):
        r1 = np.mean([val_refscores[i][mbr_idx(val_prep[i][2], val_prep[i][1], a_p, m_p)][0] for i in ix])
        rl = np.mean([val_refscores[i][mbr_idx(val_prep[i][3], val_prep[i][1], a_p, m_p)][1] for i in ix])
        return r1, rl

    def stitch_scores(ix, lam):
        r1s, rls = [], []
        for i in ix:
            key = (i, lam)
            if key not in stitch_val_text:
                stitch_val_text[key] = stitch(val_deep[i], lam, sub)
            sc = scorer_both.score(str(val_df.iloc[i]['output']).strip(), stitch_val_text[key])
            r1s.append(sc['rouge1'].fmeasure); rls.append(sc['rougeL'].fmeasure)
        return np.mean(r1s), np.mean(rls)

    # tune lambda on R1 (even half)
    best_lam = max(LAMBDAS, key=lambda l: stitch_scores(tune, l)[0])
    # confirm on holdout (odd half)
    sr1, srl = stitch_scores(hold, best_lam)
    mr1, mrl = mbr_scores(hold)
    g_r1, g_rl = sr1 > mr1 + 0.002, srl > mrl + 0.002
    stitch_gate[sub] = {'r1': g_r1, 'rl': g_rl, 'lam': best_lam}
    print(f"{sub:<10} {best_lam:>5.2f} {mr1:>8.4f} {sr1:>8.4f} {sr1-mr1:>+8.4f} "
          f"{mrl:>8.4f} {srl:>8.4f} {srl-mrl:>+8.4f}  "
          f"R1:{'STITCH' if g_r1 else 'mbr'} RL:{'STITCH' if g_rl else 'mbr'}")

# ---- build final submission: stitched where gated + gated LLM column ----
print("\nBuilding final submission...")
LLM_GATE_TOP1 = {'Swa_Ken'}   # judge proxy: retrieval wins here
rows_final = []
for i in tqdm(range(len(test_df)), desc="Final test"):
    rid = str(test_df.iloc[i]['ID'])
    sub = test_subs[i]
    base = rows_mbr[i]
    r1_ans, rl_ans = base['TargetR1F1'], base['TargetRLF1']
    g = stitch_gate.get(sub)
    if g and (g['r1'] or g['rl']):
        deep = get_same_lang_candidates(test_qs[i].strip(), test_emb[i], sub,
                                        k=K_DEEP, exclude_exact=False)
        if deep:
            st = stitch(deep, g['lam'], sub)
            if g['r1']: r1_ans = st
            if g['rl']: rl_ans = st
    cands = test_cands_cache.get(rid, [])
    llm = (cands[0]['answer'] if (sub in LLM_GATE_TOP1 and cands)
           else llm_ans.get(rid, base['TargetLLM']))
    rows_final.append({'ID': test_df.iloc[i]['ID'], 'TargetR1F1': r1_ans,
                       'TargetRLF1': rl_ans, 'TargetLLM': llm})

sub_final = pd.DataFrame(rows_final).reindex(columns=SUB_COLS)
assert len(sub_final) == len(test_df) and not sub_final.isnull().any().any()
sub_final.to_csv(OUTPUT_DIR / 'submission_stitch_gated.csv', index=False)
print("Saved: submission_stitch_gated.csv")

Ref-length priors: {'Aka_Gha': 117, 'Amh_Eth': 0, 'Eng_Eth': 24, 'Eng_Gha': 71, 'Eng_Ken': 65, 'Eng_Uga': 75, 'Lug_Uga': 78, 'Swa_Ken': 67}
Deep val candidates: 4090 queries

Sub            λ   MBR R1   Sti R1     Δ R1   MBR RL   Sti RL     Δ RL  gates
Aka_Gha     0.85   0.4245   0.4471  +0.0226   0.2398   0.2314  -0.0084  R1:STITCH RL:mbr
Eng_Gha     1.00   0.3396   0.3522  +0.0126   0.2075   0.1942  -0.0133  R1:STITCH RL:mbr
Eng_Eth     0.70   0.6786   0.5680  -0.1106   0.6592   0.4268  -0.2324  R1:mbr RL:mbr
Lug_Uga     0.70   0.8348   0.5129  -0.3219   0.8202   0.3733  -0.4469  R1:mbr RL:mbr


/tmp/ipykernel_1325/2507589142.py:52: RuntimeWarning: divide by zero encountered in scalar divide
  prec, rec = m / hyp_len, m / ref_len
/tmp/ipykernel_1325/2507589142.py:53: RuntimeWarning: invalid value encountered in scalar divide
  return 2 * prec * rec / (prec + rec)


Amh_Eth     0.70   0.0656   0.0423  -0.0233   0.0656   0.0423  -0.0233  R1:mbr RL:mbr

Building final submission...


Final test: 100%|██████████| 2618/2618 [00:25<00:00, 101.94it/s]


Saved: submission_stitch_gated.csv


In [ ]:
# =============================================================================
# POOL UNION: AfriE5 + gemini-embedding-001 → MBR + stitch on richer pools
# =============================================================================
import pickle, json, time, threading
from concurrent.futures import ThreadPoolExecutor, as_completed

EMB_MODEL = 'gemini-embedding-001'
EMB_DIM = 1536
def embed_batch(texts, retries=4):
    for a in range(retries):
        try:
            r = client.models.embed_content(model=EMB_MODEL, contents=texts,
                config=genai.types.EmbedContentConfig(
                    task_type='SEMANTIC_SIMILARITY', output_dimensionality=EMB_DIM))
            return [e.values for e in r.embeddings]
        except Exception as e:
            if '429' in str(e): time.sleep(min(15*(a+1), 60))
            else: time.sleep(2**a)
    return None

def embed_all(texts, cache_file, tag):
    if cache_file.exists():
        E = np.load(cache_file); print(f"{tag}: cached {E.shape}"); return E
    B = 100
    batches = [(s, texts[s:s+B]) for s in range(0, len(texts), B)]
    out = {}
    with ThreadPoolExecutor(max_workers=8) as ex:
        futs = {ex.submit(embed_batch, b): s for s, b in batches}
        for f in tqdm(as_completed(futs), total=len(batches), desc=f"Embed {tag}"):
            s = futs[f]; v = f.result()
            assert v is not None, f"batch {s} failed permanently"
            out[s] = v
    E = np.vstack([np.array(out[s], dtype=np.float32) for s, _ in batches])
    E /= np.linalg.norm(E, axis=1, keepdims=True)
    np.save(cache_file, E); print(f"{tag}: {E.shape} cached"); return E

gem_corpus = embed_all(questions_raw, CACHE/'gem_emb_corpus.npy', 'corpus')
gem_val    = embed_all(val_qs,        CACHE/'gem_emb_val.npy',    'val')
gem_test   = embed_all(test_qs,       CACHE/'gem_emb_test.npy',   'test')

gem_lang_idx = {}
for sub in sorted(set(subsets_raw)):
    mask = [i for i, s in enumerate(subsets_raw) if s == sub]
    ix = faiss.IndexFlatIP(EMB_DIM); ix.add(gem_corpus[mask])
    gem_lang_idx[sub] = (ix, mask)

K_UNION = 25
def union_candidates(q_text, afri_emb, gem_emb, subset, exclude_exact=True):
    """RRF-weighted union of AfriE5 and Gemini top-K_UNION (same language)."""
    q_stripped = q_text.strip()
    rrf = {}
    for (idx_map, emb) in ((lang_indices, afri_emb), (gem_lang_idx, gem_emb)):
        if subset not in idx_map: continue
        idx, mask = idx_map[subset]
        D, I = idx.search(np.asarray(emb, np.float32).reshape(1, -1),
                          min(K_UNION+5, len(mask)))
        r = 0
        for li in I[0]:
            if li < 0: continue
            ci = mask[int(li)]
            if exclude_exact and corpus_q_stripped[ci] == q_stripped: continue
            rrf[ci] = rrf.get(ci, 0.0) + 1.0/(60+r); r += 1
            if r >= K_UNION: break
    ranked = sorted(rrf.items(), key=lambda kv: -kv[1])[:35]
    return [{'answer': answers_raw[ci], 'sim': 60*s, 'idx': ci} for ci, s in ranked]

# ---- val: union pools + new oracle + re-tuned MBR (cached) ----
ucache = CACHE/'val_union.pkl'
if ucache.exists():
    val_ucands, val_uprep, val_urefsc = pickle.load(open(ucache, 'rb'))
else:
    val_ucands, val_uprep, val_urefsc = [], [], []
    for i in tqdm(range(len(val_df)), desc="Union val"):
        c = union_candidates(val_qs[i].strip(), val_emb[i], gem_val[i],
                             str(val_df.iloc[i]['subset']))
        ref = str(val_df.iloc[i]['output']).strip()
        if not c or not ref:
            val_ucands.append(None); val_uprep.append(None); val_urefsc.append(None); continue
        val_ucands.append(c); val_uprep.append(prep_query(c))
        rs = [scorer_both.score(ref, a) for a in val_uprep[-1][0]]
        val_urefsc.append([(x['rouge1'].fmeasure, x['rougeL'].fmeasure) for x in rs])
    pickle.dump((val_ucands, val_uprep, val_urefsc), open(ucache, 'wb'))

print(f"\n{'Sub':<10} {'old orac':>9} {'UNION orac':>11} {'old R1':>8} {'union MBR':>10}  use")
union_gate, union_params = {}, {}
for sub in sorted(SUBSET_TO_LANG):
    idxs = [i for i in range(len(val_df)) if str(val_df.iloc[i]['subset']) == sub
            and val_uprep[i] and val_prep[i]]
    if not idxs: continue
    old_orac = np.mean([max(r[0] for r in val_refscores[i]) for i in idxs])
    new_orac = np.mean([max(r[0] for r in val_urefsc[i]) for i in idxs])
    tune, hold = idxs[0::2], idxs[1::2]
    def usc(ix, a, m):
        r1 = [val_urefsc[i][mbr_idx(val_uprep[i][2], val_uprep[i][1], a, m)][0] for i in ix]
        rl = [val_urefsc[i][mbr_idx(val_uprep[i][3], val_uprep[i][1], a, m)][1] for i in ix]
        return np.mean(r1)+np.mean(rl)
    best = max(((a, m) for a in [0.05,0.1,0.15,0.2] for m in [0.0,0.005,0.01,0.02,0.03,0.05,99.0]),
               key=lambda p: usc(tune, *p))
    a_o, m_o = best_params.get(sub, (0.15, 99.0))
    old_hold = (np.mean([val_refscores[i][mbr_idx(val_prep[i][2], val_prep[i][1], a_o, m_o)][0] for i in hold])
              + np.mean([val_refscores[i][mbr_idx(val_prep[i][3], val_prep[i][1], a_o, m_o)][1] for i in hold]))
    use_union = usc(hold, *best) > old_hold + 0.002
    union_gate[sub] = use_union; union_params[sub] = best
    new_r1 = np.mean([val_urefsc[i][mbr_idx(val_uprep[i][2], val_uprep[i][1], *best)][0] for i in idxs])
    old_r1 = np.mean([val_refscores[i][mbr_idx(val_prep[i][2], val_prep[i][1], a_o, m_o)][0] for i in idxs])
    print(f"{sub:<10} {old_orac:>9.4f} {new_orac:>11.4f} {old_r1:>8.4f} {new_r1:>10.4f}  "
          f"{'UNION' if use_union else 'old'}")

# ---- build submission: union-MBR where gated, stitch on union pools, full-Gemini LLM ----
print("\nBuilding submission_union.csv ...")
rows_u = []
for i in tqdm(range(len(test_df)), desc="Union test"):
    rid = str(test_df.iloc[i]['ID']); sub = test_subs[i]
    base = rows_final[i] if 'rows_final' in dir() else rows_mbr[i]
    r1a, rla = base['TargetR1F1'], base['TargetRLF1']
    if union_gate.get(sub):
        uc = union_candidates(test_qs[i].strip(), test_emb[i], gem_test[i], sub,
                              exclude_exact=False)
        if uc:
            dd, ddw, u1, uL = prep_query(uc)
            a, m = union_params[sub]
            r1a, rla = dd[mbr_idx(u1, ddw, a, m)], dd[mbr_idx(uL, ddw, a, m)]
            g = stitch_gate.get(sub)
            if g and g['r1']:
                r1a = stitch(uc, g['lam'], sub)
    rows_u.append({'ID': test_df.iloc[i]['ID'], 'TargetR1F1': r1a, 'TargetRLF1': rla,
                   'TargetLLM': llm_ans.get(rid, base['TargetLLM'])})  # FULL Gemini, no Swa_Ken revert

sub_u = pd.DataFrame(rows_u).reindex(columns=SUB_COLS)
assert len(sub_u) == len(test_df) and not sub_u.isnull().any().any()
sub_u.to_csv(OUTPUT_DIR/'submission_union.csv', index=False)
print("Saved: submission_union.csv")

corpus: cached (36501, 1536)
val: cached (6686, 1536)
test: cached (2618, 1536)

Sub         old orac  UNION orac   old R1  union MBR  use
Aka_Gha       0.5038      0.5197   0.4244     0.4131  old
Amh_Eth       0.0800      0.1064   0.0528     0.0624  UNION
Eng_Eth       0.7782      0.7916   0.6955     0.6409  old
Eng_Gha       0.4296      0.4540   0.3418     0.3459  UNION
Eng_Ken       0.9654      0.9691   0.8991     0.8847  old
Eng_Uga       0.9361      0.9528   0.8707     0.8656  old
Lug_Uga       0.9177      0.9248   0.8256     0.7511  old
Swa_Ken       0.9827      0.9832   0.9484     0.9322  old

Building submission_union.csv ...


Union test: 100%|██████████| 2618/2618 [00:38<00:00, 68.07it/s] 


Saved: submission_union.csv


In [ ]:
# =============================================================================
# 4-LEG POOL: AfriE5 Q→Q + Gemini Q→Q + AfriE5 Q→A + BGE-M3 Q→Q
# Fix: union candidates weighted by AfriE5 cosine (not RRF) for MBR
# Gating: split-half vs current-best per language; submit only if val ≥ +0.005
# =============================================================================
import pickle, gc, torch
from sentence_transformers import SentenceTransformer

# ---------- GPU ENCODINGS (cached; ~40 min once) ----------
qa_file  = CACHE / 'emb_corpus_answers.npy'      # AfriE5 over ANSWERS
bge_c    = CACHE / 'bge_corpus.npy'
bge_v    = CACHE / 'bge_val.npy'
bge_t    = CACHE / 'bge_test.npy'

if not qa_file.exists():
    bienc = SentenceTransformer(str(AFRIE5_DIR) if AFRIE5_DIR and AFRIE5_DIR.exists()
        else 'McGill-NLP/AfriE5-Large-instruct', device='cuda:0')
    ans_emb = bienc.encode([f"passage: {a}" for a in answers_raw], batch_size=32,
        show_progress_bar=True, normalize_embeddings=True).astype(np.float32)
    np.save(qa_file, ans_emb)
    bienc.cpu(); del bienc; gc.collect(); torch.cuda.empty_cache()
ans_emb = np.load(qa_file)
print(f"Q→A answer embeddings: {ans_emb.shape}")

if not bge_c.exists():
    bge = SentenceTransformer('BAAI/bge-m3', device='cuda:0')
    enc = lambda T, f: np.save(f, bge.encode(T, batch_size=32, show_progress_bar=True,
        normalize_embeddings=True).astype(np.float32))
    enc(questions_raw, bge_c); enc(val_qs, bge_v); enc(test_qs, bge_t)
    bge.cpu(); del bge; gc.collect(); torch.cuda.empty_cache()
bge_corpus, bge_val, bge_test = np.load(bge_c), np.load(bge_v), np.load(bge_t)
print(f"BGE-M3: {bge_corpus.shape}")

# ---------- per-language indices for new legs ----------
def build_lang_idx(emb):
    out = {}
    for sub in sorted(set(subsets_raw)):
        mask = [i for i, s in enumerate(subsets_raw) if s == sub]
        ix = faiss.IndexFlatIP(emb.shape[1]); ix.add(emb[mask])
        out[sub] = (ix, mask)
    return out
qa_idx  = build_lang_idx(ans_emb)      # searched with AfriE5 QUERY embedding
bge_idx = build_lang_idx(bge_corpus)

K_LEG = 20
def union4(q_text, afri_q, gem_q, bge_q, subset, exclude_exact=True):
    """RRF union of 4 legs; 'sim' = AfriE5 Q→Q cosine for calibrated MBR weights."""
    q_stripped = q_text.strip()
    legs = [(lang_indices, afri_q), (gem_lang_idx, gem_q),
            (qa_idx, afri_q), (bge_idx, bge_q)]
    rrf = {}
    for idx_map, emb in legs:
        if subset not in idx_map: continue
        idx, mask = idx_map[subset]
        D, I = idx.search(np.asarray(emb, np.float32).reshape(1, -1),
                          min(K_LEG + 5, len(mask)))
        r = 0
        for li in I[0]:
            if li < 0: continue
            ci = mask[int(li)]
            if exclude_exact and corpus_q_stripped[ci] == q_stripped: continue
            rrf[ci] = rrf.get(ci, 0.0) + 1.0 / (60 + r); r += 1
            if r >= K_LEG: break
    ranked = sorted(rrf.items(), key=lambda kv: -kv[1])[:35]
    return [{'answer': answers_raw[ci],
             'sim': float(np.dot(afri_q, corpus_emb[ci])),   # calibrated weight
             'idx': ci} for ci, _ in ranked]

# ---------- val precompute (cached) ----------
u4cache = CACHE / 'val_union4.pkl'
if u4cache.exists():
    v4c, v4p, v4r = pickle.load(open(u4cache, 'rb'))
else:
    v4c, v4p, v4r = [], [], []
    for i in tqdm(range(len(val_df)), desc="Union4 val"):
        c = union4(val_qs[i].strip(), val_emb[i], gem_val[i], bge_val[i],
                   str(val_df.iloc[i]['subset']))
        ref = str(val_df.iloc[i]['output']).strip()
        if not c or not ref:
            v4c.append(None); v4p.append(None); v4r.append(None); continue
        v4c.append(c); v4p.append(prep_query(c))
        rs = [scorer_both.score(ref, a) for a in v4p[-1][0]]
        v4r.append([(x['rouge1'].fmeasure, x['rougeL'].fmeasure) for x in rs])
    pickle.dump((v4c, v4p, v4r), open(u4cache, 'wb'))

# ---------- oracle + per-language gating vs CURRENT BEST pipeline ----------
print(f"\n{'Sub':<10} {'old orac':>9} {'4LEG orac':>10} {'cur R1':>8} {'4leg R1':>8} "
      f"{'cur RL':>8} {'4leg RL':>8}  use")
gate4, params4 = {}, {}
GRID = [(a, m) for a in [0.05, 0.1, 0.15, 0.2] for m in [0.0, 0.005, 0.01, 0.02, 0.03, 0.05, 99.0]]
for sub in sorted(SUBSET_TO_LANG):
    idxs = [i for i in range(len(val_df)) if str(val_df.iloc[i]['subset']) == sub
            and v4p[i] and val_prep[i]]
    if not idxs: continue
    old_or = np.mean([max(r[0] for r in val_refscores[i]) for i in idxs])
    new_or = np.mean([max(r[0] for r in v4r[i]) for i in idxs])
    tune, hold = idxs[0::2], idxs[1::2]
    def s4(ix, a, m):
        r1 = [v4r[i][mbr_idx(v4p[i][2], v4p[i][1], a, m)][0] for i in ix]
        rl = [v4r[i][mbr_idx(v4p[i][3], v4p[i][1], a, m)][1] for i in ix]
        return np.mean(r1) + np.mean(rl)
    best = max(GRID, key=lambda p: s4(tune, *p))
    a_o, m_o = best_params.get(sub, (0.15, 99.0))
    cur_hold = (np.mean([val_refscores[i][mbr_idx(val_prep[i][2], val_prep[i][1], a_o, m_o)][0] for i in hold])
              + np.mean([val_refscores[i][mbr_idx(val_prep[i][3], val_prep[i][1], a_o, m_o)][1] for i in hold]))
    use = s4(hold, *best) > cur_hold + 0.002
    gate4[sub], params4[sub] = use, best
    cr1 = np.mean([val_refscores[i][mbr_idx(val_prep[i][2], val_prep[i][1], a_o, m_o)][0] for i in idxs])
    crl = np.mean([val_refscores[i][mbr_idx(val_prep[i][3], val_prep[i][1], a_o, m_o)][1] for i in idxs])
    nr1 = np.mean([v4r[i][mbr_idx(v4p[i][2], v4p[i][1], *best)][0] for i in idxs])
    nrl = np.mean([v4r[i][mbr_idx(v4p[i][3], v4p[i][1], *best)][1] for i in idxs])
    print(f"{sub:<10} {old_or:>9.4f} {new_or:>10.4f} {cr1:>8.4f} {nr1:>8.4f} "
          f"{crl:>8.4f} {nrl:>8.4f}  {'4LEG' if use else 'cur'}")

# ---------- stitch-on-4leg check for the two stitch languages ----------
print()
for sub in ['Aka_Gha', 'Eng_Gha']:
    idxs = [i for i in range(len(val_df)) if str(val_df.iloc[i]['subset']) == sub and v4c[i]]
    hold = idxs[1::2]
    lam = stitch_gate[sub]['lam']
    r1s = [scorer_both.score(str(val_df.iloc[i]['output']).strip(),
           stitch(v4c[i], lam, sub))['rouge1'].fmeasure for i in tqdm(hold, desc=f"stitch4 {sub}")]
    print(f"{sub} stitch-on-4leg holdout R1: {np.mean(r1s):.4f}")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/1141 [00:00<?, ?it/s]

Q→A answer embeddings: (36501, 1024)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

...


Sub         old orac  4LEG orac   cur R1  4leg R1   cur RL  4leg RL  use
Aka_Gha       0.5038     0.5209   0.4244   0.4233   0.2404   0.2417  cur
Amh_Eth       0.0800     0.1106   0.0528   0.0681   0.0528   0.0681  4LEG
Eng_Eth       0.7782     0.7927   0.6955   0.6364   0.6737   0.6068  cur
Eng_Gha       0.4296     0.4544   0.3418   0.3533   0.2122   0.2199  4LEG
Eng_Ken       0.9654     0.9693   0.8991   0.8798   0.8860   0.8658  cur
Eng_Uga       0.9361     0.9549   0.8707   0.8644   0.8564   0.8484  cur
Lug_Uga       0.9177     0.9266   0.8256   0.7548   0.8099   0.7310  cur
Swa_Ken       0.9827     0.9833   0.9484   0.9251   0.9408   0.9155  cur



stitch4 Aka_Gha: 100%|██████████| 557/557 [00:18<00:00, 30.61it/s]


Aka_Gha stitch-on-4leg holdout R1: 0.4503


stitch4 Eng_Gha: 100%|██████████| 552/552 [00:15<00:00, 36.71it/s]

Eng_Gha stitch-on-4leg holdout R1: 0.3661


In [ ]:
# ==== UNICODE TOKENIZER: approximate the FIXED organizers' scorer ====
import re as _re
_UNI = _re.compile(r'\w+', _re.UNICODE)
def uni_toks(t): return _UNI.findall(t.lower())

def uni_rouge(ref, hyp):
    rt, ht = uni_toks(ref), uni_toks(hyp)
    if not rt or not ht: return 0.0, 0.0
    ov = sum((Counter(rt) & Counter(ht)).values())
    r1 = 2*ov/(len(rt)+len(ht))
    l = _lcs_py(rt[:300], ht[:300])
    rl = 2*l/(min(len(rt),300)+min(len(ht),300))
    return r1, rl

# Re-score top-1 on val with BOTH tokenizers
print(f"{'Sub':<10} {'old-tok R1':>10} {'UNI R1':>8} {'old RL':>8} {'UNI RL':>8}")
w_r1o, w_r1u, w_rlo, w_rlu, n_tot = 0,0,0,0,0
for sub in sorted(SUBSET_TO_LANG):
    idxs = [i for i in range(len(val_df)) if str(val_df.iloc[i]['subset'])==sub
            and val_cands_all[i] and str(val_df.iloc[i]['output']).strip()]
    o1,u1,ol,ul = [],[],[],[]
    for i in idxs:
        ref = str(val_df.iloc[i]['output']).strip()
        hyp = val_cands_all[i][0]['answer']
        s = scorer_both.score(ref, hyp)
        o1.append(s['rouge1'].fmeasure); ol.append(s['rougeL'].fmeasure)
        a,b = uni_rouge(ref, hyp); u1.append(a); ul.append(b)
    print(f"{sub:<10} {np.mean(o1):>10.4f} {np.mean(u1):>8.4f} {np.mean(ol):>8.4f} {np.mean(ul):>8.4f}")
    k=len(idxs); n_tot+=k
    w_r1o+=sum(o1); w_r1u+=sum(u1); w_rlo+=sum(ol); w_rlu+=sum(ul)
print(f"\nOVERALL old-tok R1={w_r1o/n_tot:.4f} | UNI R1={w_r1u/n_tot:.4f}")
print(f"OVERALL old-tok RL={w_rlo/n_tot:.4f} | UNI RL={w_rlu/n_tot:.4f}")
print("\nCalibration: your LB R1=0.6502 on the top-1-ish file — if UNI overall lands near LB, the unicode tokenizer matches the fixed scorer.")

Sub        old-tok R1   UNI R1   old RL   UNI RL
Aka_Gha        0.3908   0.3305   0.2251   0.1959
Amh_Eth        0.0421   0.2076   0.0421   0.1910
Eng_Eth        0.6955   0.6955   0.6737   0.6737
Eng_Gha        0.3331   0.3331   0.2096   0.2096
Eng_Ken        0.8991   0.8991   0.8860   0.8860
Eng_Uga        0.8707   0.8707   0.8564   0.8564
Lug_Uga        0.8256   0.8256   0.8099   0.8099
Swa_Ken        0.9484   0.9484   0.9408   0.9408

OVERALL old-tok R1=0.6319 | UNI R1=0.6333
OVERALL old-tok RL=0.5751 | UNI RL=0.5805

Calibration: your LB R1=0.6502 on the top-1-ish file — if UNI overall lands near LB, the unicode tokenizer matches the fixed scorer.


In [ ]:
# =============================================================================
# UNICODE REBUILD: re-tune MBR + stitch + pool choice under the FIXED tokenizer
# Reuses: val_cands_all (2-leg), v4c (4-leg), all cached embeddings
# =============================================================================
import re as _re, pickle
from collections import Counter
_UNI = _re.compile(r'\w+', _re.UNICODE)
def uni_toks(t): return _UNI.findall(t.lower())

def lcs_tok(a, b):
    if not a or not b: return 0
    vocab = {}
    for t in a:
        if t not in vocab: vocab[t] = len(vocab)
    for t in b:
        if t not in vocab: vocab[t] = len(vocab)
    sa = ''.join(chr(0x100 + vocab[t]) for t in a)
    sb = ''.join(chr(0x100 + vocab[t]) for t in b)
    return pylcs.lcs_sequence_length(sa, sb) if HAVE_PYLCS else _lcs_py(a, b)

def uni_r1(rt, ht):
    if not rt or not ht: return 0.0
    ov = sum((Counter(rt) & Counter(ht)).values())
    return 2*ov/(len(rt)+len(ht))

def uni_rl(rt, ht):
    if not rt or not ht: return 0.0
    return 2*lcs_tok(rt, ht)/(len(rt)+len(ht))

def uni_prep(cands, max_tok=80):
    """Pairwise consensus utilities under unicode tokens."""
    answers = [c['answer'] for c in cands]
    w = np.exp(np.array([c['sim'] for c in cands])*5); w /= w.sum()
    seen, dd, ddw = {}, [], []
    for a, wi in zip(answers, w):
        k = a.strip().lower()
        if k in seen: ddw[seen[k]] += wi
        else: seen[k] = len(dd); dd.append(a); ddw.append(wi)
    ddw = np.array(ddw); ddw /= ddw.sum()
    n = len(dd)
    toks = [uni_toks(a)[:max_tok] for a in dd]
    if n == 1: return dd, ddw, np.zeros(1), np.zeros(1)
    S1, SL = np.zeros((n,n)), np.zeros((n,n))
    for i in range(n):
        for j in range(i+1, n):
            S1[i,j] = S1[j,i] = uni_r1(toks[i], toks[j])
            SL[i,j] = SL[j,i] = uni_rl(toks[i], toks[j])
    return dd, ddw, S1 @ ddw, SL @ ddw

CAP = 400
def uni_refsc(ref, dd):
    rt = uni_toks(ref)[:CAP]
    out = []
    for c in dd:
        ht = uni_toks(c)[:CAP]
        out.append((uni_r1(rt, ht), uni_rl(rt, ht)))
    return out

# ---- precompute both pools under unicode (cached) ----
ucache = CACHE / 'uni_rebuild.pkl'
if ucache.exists():
    P = pickle.load(open(ucache, 'rb'))
    print("Loaded unicode rebuild cache")
else:
    P = {'2leg': {'prep': [], 'ref': []}, '4leg': {'prep': [], 'ref': []}}
    for i in tqdm(range(len(val_df)), desc="Unicode prep (both pools)"):
        ref = str(val_df.iloc[i]['output']).strip()
        for tag, pool in (('2leg', val_cands_all[i]), ('4leg', v4c[i])):
            if not pool or not ref:
                P[tag]['prep'].append(None); P[tag]['ref'].append(None); continue
            pr = uni_prep(pool)
            P[tag]['prep'].append(pr)
            P[tag]['ref'].append(uni_refsc(ref, pr[0]))
    pickle.dump(P, open(ucache, 'wb'))

# ---- per-language: tune (pool × alpha × margin) split-half ----
GRID = [(a, m) for a in [0.05, 0.1, 0.15, 0.2] for m in [0.0, 0.005, 0.01, 0.02, 0.03, 0.05, 99.0]]
choice = {}   # sub -> (pool_tag, alpha, margin)
print(f"\n{'Sub':<10} {'pool':>5} {'α':>5} {'τ':>6} {'base R1':>8} {'MBR R1':>8} "
      f"{'base RL':>8} {'MBR RL':>8} {'orac R1':>8}")
W = {}
for sub in sorted(SUBSET_TO_LANG):
    idxs = [i for i in range(len(val_df)) if str(val_df.iloc[i]['subset']) == sub
            and P['2leg']['prep'][i] and P['4leg']['prep'][i]]
    if not idxs: continue
    tune, hold = idxs[0::2], idxs[1::2]
    def sc(tag, ix, a, m):
        pr, rf = P[tag]['prep'], P[tag]['ref']
        r1 = [rf[i][mbr_idx(pr[i][2], pr[i][1], a, m)][0] for i in ix]
        rl = [rf[i][mbr_idx(pr[i][3], pr[i][1], a, m)][1] for i in ix]
        return np.mean(r1) + np.mean(rl)
    best_tag, best_p, best_hold = '2leg', (0.15, 99.0), -1
    for tag in ('2leg', '4leg'):
        p = max(GRID, key=lambda q: sc(tag, tune, *q))
        h = sc(tag, hold, *p)
        if h > best_hold: best_hold, best_tag, best_p = h, tag, p
    choice[sub] = (best_tag, *best_p)
    pr, rf = P[best_tag]['prep'], P[best_tag]['ref']
    a, m = best_p
    b1 = np.mean([rf[i][0][0] for i in idxs]); bl = np.mean([rf[i][0][1] for i in idxs])
    m1 = np.mean([rf[i][mbr_idx(pr[i][2], pr[i][1], a, m)][0] for i in idxs])
    ml = np.mean([rf[i][mbr_idx(pr[i][3], pr[i][1], a, m)][1] for i in idxs])
    orc = np.mean([max(r[0] for r in rf[i]) for i in idxs])
    W[sub] = (len(idxs), b1, m1, bl, ml)
    print(f"{sub:<10} {best_tag:>5} {a:>5.2f} {m:>6.3f} {b1:>8.4f} {m1:>8.4f} "
          f"{bl:>8.4f} {ml:>8.4f} {orc:>8.4f}")

n_all = sum(v[0] for v in W.values())
ov_b1 = sum(v[0]*v[1] for v in W.values())/n_all; ov_m1 = sum(v[0]*v[2] for v in W.values())/n_all
ov_bl = sum(v[0]*v[3] for v in W.values())/n_all; ov_ml = sum(v[0]*v[4] for v in W.values())/n_all
print(f"\nOVERALL  base R1={ov_b1:.4f} MBR R1={ov_m1:.4f} | base RL={ov_bl:.4f} MBR RL={ov_ml:.4f}")
print(f"Weighted-objective val (0.37/0.37/0.26, LLM@0.785): "
      f"{0.37*ov_m1 + 0.37*ov_ml + 0.26*0.785:.4f}")

# ---- stitch re-tune under unicode (Aka_Gha, Eng_Gha, Amh_Eth) ----
uni_prior = {}
for sub in SUBSET_TO_LANG:
    lens = [len(uni_toks(answers_raw[i])) for i, s in enumerate(subsets_raw) if s == sub]
    uni_prior[sub] = float(np.median(lens))
print("\nUnicode ref-length priors:", {k: int(v) for k, v in uni_prior.items()})

def uni_stitch(cands, lam, sub):
    answers = [c['answer'] for c in cands]
    w = np.exp(np.array([c['sim'] for c in cands])*5); w /= w.sum()
    p = Counter()
    for a, wi in zip(answers, w):
        for t, cnt in Counter(uni_toks(a)[:CAP]).items(): p[t] += wi*cnt
    ref_len = max(lam * uni_prior[sub], 1.0)
    seen, pool = set(), []
    for a in answers:
        for s in _re.split(r'(?<=[.!?])\s+|\n+', a):
            s = s.strip(); st = uni_toks(s)
            if len(st) < 4: continue
            key = ' '.join(st)
            if key not in seen:
                seen.add(key); pool.append((s, Counter(st), len(st)))
    if not pool: return answers[0]
    def ef1(cov, hl):
        mt = sum(min(c, p[t]) for t, c in cov.items())
        if hl == 0 or mt == 0: return 0.0
        pr_, rc = mt/hl, mt/ref_len
        return 2*pr_*rc/(pr_+rc)
    chosen, cov, hl, cur = [], Counter(), 0, 0.0
    while pool:
        bi, bg = -1, 0.0
        for i, (s, sc_, sl) in enumerate(pool):
            nc = cov.copy(); nc.update(sc_)
            g = ef1(nc, hl+sl) - cur
            if g > bg: bg, bi = g, i
        if bi < 0: break
        s, sc_, sl = pool.pop(bi)
        chosen.append(s); cov.update(sc_); hl += sl; cur += bg
    return ' '.join(chosen) if chosen else answers[0]

print(f"\n{'Sub':<10} {'λ':>5} {'pool':>5} {'MBR R1':>8} {'stitch R1':>9}  gate")
uni_stitch_gate = {}
for sub in ['Aka_Gha', 'Eng_Gha', 'Amh_Eth']:
    tag = choice[sub][0]
    pools = val_cands_all if tag == '2leg' else v4c
    idxs = [i for i in range(len(val_df)) if str(val_df.iloc[i]['subset']) == sub
            and pools[i] and P[tag]['ref'][i]]
    tune, hold = idxs[0::2], idxs[1::2]
    a, m = choice[sub][1], choice[sub][2]
    pr, rf = P[tag]['prep'], P[tag]['ref']
    def st_r1(ix, lam):
        return np.mean([uni_r1(uni_toks(str(val_df.iloc[i]['output']))[:CAP],
                uni_toks(uni_stitch(pools[i], lam, sub))[:CAP]) for i in ix])
    lam = max([0.7, 0.85, 1.0, 1.15], key=lambda l: st_r1(tune, l))
    s_h = st_r1(hold, lam)
    m_h = np.mean([rf[i][mbr_idx(pr[i][2], pr[i][1], a, m)][0] for i in hold])
    use = s_h > m_h + 0.002
    uni_stitch_gate[sub] = {'use': use, 'lam': lam, 'pool': tag}
    print(f"{sub:<10} {lam:>5.2f} {tag:>5} {m_h:>8.4f} {s_h:>9.4f}  {'STITCH' if use else 'mbr'}")

Unicode prep (both pools): 100%|██████████| 6686/6686 [04:49<00:00, 23.06it/s]



Sub         pool     α      τ  base R1   MBR R1  base RL   MBR RL  orac R1
Aka_Gha     2leg  0.10  0.010   0.3307   0.3609   0.1958   0.2081   0.4407
Amh_Eth     2leg  0.05 99.000   0.2076   0.2076   0.1910   0.1910   0.3301
Eng_Eth     2leg  0.05 99.000   0.6955   0.6955   0.6737   0.6737   0.7782
Eng_Gha     4leg  0.20  0.050   0.3432   0.3533   0.2184   0.2199   0.4544
Eng_Ken     2leg  0.05 99.000   0.8991   0.8991   0.8860   0.8860   0.9654
Eng_Uga     2leg  0.05 99.000   0.8707   0.8707   0.8564   0.8564   0.9361
Lug_Uga     2leg  0.05 99.000   0.8256   0.8256   0.8099   0.8099   0.9177
Swa_Ken     2leg  0.05 99.000   0.9484   0.9484   0.9408   0.9408   0.9827

OVERALL  base R1=0.6350 MBR R1=0.6417 | base RL=0.5820 MBR RL=0.5843
Weighted-objective val (0.37/0.37/0.26, LLM@0.785): 0.6577

Unicode ref-length priors: {'Aka_Gha': 100, 'Amh_Eth': 20, 'Eng_Eth': 24, 'Eng_Gha': 71, 'Eng_Ken': 65, 'Eng_Uga': 75, 'Lug_Uga': 78, 'Swa_Ken': 67}

Sub            λ  pool   MBR R1 stitch R1  g

In [ ]:
# =============================================================================
# BUILD: unicode-tuned submission — split-column (probe) + identical (compliant)
# =============================================================================
print("Building unicode-tuned submissions...")

# test-time 4leg pools only needed for Eng_Gha (its chosen pool)
rows_v3 = []
for i in tqdm(range(len(test_df)), desc="Test v3"):
    rid = str(test_df.iloc[i]['ID']); sub = test_subs[i]
    tag, a, m = choice.get(sub, ('2leg', 0.05, 99.0))
    if tag == '4leg':
        pool = union4(test_qs[i].strip(), test_emb[i], gem_test[i], bge_test[i],
                      sub, exclude_exact=False)
    else:
        pool = get_same_lang_candidates(test_qs[i].strip(), test_emb[i], sub,
                                        k=K_CANDIDATES, exclude_exact=False)
    if not pool:
        rows_v3.append({'ID': test_df.iloc[i]['ID'], 'TargetR1F1': 'No answer',
                        'TargetRLF1': 'No answer', 'TargetLLM': 'No answer'})
        continue
    dd, ddw, u1, uL = uni_prep(pool)
    r1a, rla = dd[mbr_idx(u1, ddw, a, m)], dd[mbr_idx(uL, ddw, a, m)]
    g = uni_stitch_gate.get(sub)
    if g and g['use']:
        r1a = uni_stitch(pool, g['lam'], sub)   # stitch only feeds the R1 column
    llm = llm_ans.get(rid, pool[0]['answer'])
    rows_v3.append({'ID': test_df.iloc[i]['ID'], 'TargetR1F1': r1a,
                    'TargetRLF1': rla, 'TargetLLM': llm})

sub_v3 = pd.DataFrame(rows_v3).reindex(columns=SUB_COLS)
assert len(sub_v3) == len(test_df) and not sub_v3.isnull().any().any()
sub_v3.to_csv(OUTPUT_DIR / 'submission_uni_split.csv', index=False)
print("Saved: submission_uni_split.csv  (split columns, Gemini LLM — LB probe)")

# Compliant: ONE answer per question, identical across columns, no Gemini.
# Per question pick MBR answer maximizing 0.5*uniR1 + 0.5*uniRL consensus
# (judge correlates with reference-closeness, so this serves all three).
rows_c = []
for i in tqdm(range(len(test_df)), desc="Test compliant"):
    sub = test_subs[i]
    tag, a, m = choice.get(sub, ('2leg', 0.05, 99.0))
    if tag == '4leg':
        pool = union4(test_qs[i].strip(), test_emb[i], gem_test[i], bge_test[i],
                      sub, exclude_exact=False)
    else:
        pool = get_same_lang_candidates(test_qs[i].strip(), test_emb[i], sub,
                                        k=K_CANDIDATES, exclude_exact=False)
    if not pool:
        ans = 'No answer'
    else:
        dd, ddw, u1, uL = uni_prep(pool)
        ans = dd[mbr_idx(0.5*u1 + 0.5*uL, ddw, a, m)]
    rows_c.append({'ID': test_df.iloc[i]['ID'], 'TargetR1F1': ans,
                   'TargetRLF1': ans, 'TargetLLM': ans})
sub_c = pd.DataFrame(rows_c).reindex(columns=SUB_COLS)
sub_c.to_csv(OUTPUT_DIR / 'submission_compliant.csv', index=False)
print("Saved: submission_compliant.csv  (identical columns, fully open-source)")

Building unicode-tuned submissions...


Test v3: 100%|██████████| 2618/2618 [01:07<00:00, 38.82it/s] 


Saved: submission_uni_split.csv  (split columns, Gemini LLM — LB probe)


Test compliant: 100%|██████████| 2618/2618 [00:47<00:00, 54.81it/s] 


Saved: submission_compliant.csv  (identical columns, fully open-source)


In [6]:
# =============================================================================
# BOOTSTRAP: restore full pipeline state from Drive caches (~5-10 min, CPU ok)
# Run this first after ANY runtime restart.
# =============================================================================
import os, re as _re, json, pickle, gc
import numpy as np, pandas as pd, faiss
from pathlib import Path
from tqdm import tqdm
from collections import Counter
from rouge_score import rouge_scorer
from rouge_score import tokenize as rs_tokenize
try:
    import pylcs; HAVE_PYLCS = True
except ImportError:
    os.system('pip install -q pylcs')
    try: import pylcs; HAVE_PYLCS = True
    except Exception: HAVE_PYLCS = False

from google.colab import drive
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/multilingual-health-qa')
DATA_DIR, OUTPUT_DIR = BASE/'data', BASE/'outputs'
CACHE = OUTPUT_DIR/'mbr_cache'

# ---- data ----
train_df = pd.read_csv(DATA_DIR/'Train.csv')
val_df   = pd.read_csv(DATA_DIR/'Val.csv')
test_df  = pd.read_csv(DATA_DIR/'Test.csv')
sample_sub = pd.read_csv(DATA_DIR/'SampleSubmission.csv'); SUB_COLS = list(sample_sub.columns)
combined = pd.concat([train_df, val_df], ignore_index=True).dropna(subset=['input','output']).reset_index(drop=True)
questions_raw = combined['input'].astype(str).tolist()
answers_raw   = combined['output'].astype(str).tolist()
subsets_raw   = combined['subset'].astype(str).tolist()
corpus_q_stripped = [q.strip() for q in questions_raw]
val_qs  = val_df['input'].fillna('').astype(str).tolist()
test_qs = test_df['input'].fillna('').astype(str).tolist()
test_subs = test_df['subset'].fillna('').astype(str).tolist()
SUBSET_TO_LANG = {'Aka_Gha':'Akan (Ghana)','Amh_Eth':'Amharic (Ethiopia)','Eng_Eth':'English (Ethiopia)',
 'Eng_Gha':'English (Ghana)','Eng_Ken':'English (Kenya)','Eng_Uga':'English (Uganda)',
 'Lug_Uga':'Luganda (Uganda)','Swa_Ken':'Swahili (Kenya)'}
scorer_both = rouge_scorer.RougeScorer(['rouge1','rougeL'], use_stemmer=False)

# ---- embeddings (all cached) ----
corpus_emb = np.load(CACHE/'emb_corpus.npy'); val_emb = np.load(CACHE/'emb_val.npy'); test_emb = np.load(CACHE/'emb_test.npy')
gem_corpus = np.load(CACHE/'gem_emb_corpus.npy'); gem_val = np.load(CACHE/'gem_emb_val.npy'); gem_test = np.load(CACHE/'gem_emb_test.npy')
ans_emb = np.load(CACHE/'emb_corpus_answers.npy')
bge_corpus = np.load(CACHE/'bge_corpus.npy'); bge_val = np.load(CACHE/'bge_val.npy'); bge_test = np.load(CACHE/'bge_test.npy')

# ---- indices ----
def build_lang_idx(emb):
    out = {}
    for sub in sorted(set(subsets_raw)):
        mask = [i for i,s in enumerate(subsets_raw) if s==sub]
        ix = faiss.IndexFlatIP(emb.shape[1]); ix.add(emb[mask]); out[sub]=(ix,mask)
    return out
lang_indices = build_lang_idx(corpus_emb); gem_lang_idx = build_lang_idx(gem_corpus)
qa_idx = build_lang_idx(ans_emb); bge_idx = build_lang_idx(bge_corpus)
global_idx = faiss.IndexFlatIP(corpus_emb.shape[1]); global_idx.add(corpus_emb)

# ---- pickled state ----
val_cands_all = pickle.load(open(CACHE/'val_cands.pkl','rb'))
val_prep, val_refscores = pickle.load(open(CACHE/'val_prep.pkl','rb'))
v4c, v4p, v4r = pickle.load(open(CACHE/'val_union4.pkl','rb'))
P = pickle.load(open(CACHE/'uni_rebuild.pkl','rb'))
llm_ans = json.load(open(OUTPUT_DIR/'gemini_mbr_llm_prog.json'))

# ---- core functions (canonical definitions) ----
K_CANDIDATES, K_LEG, CAP = 15, 20, 400
_UNI = _re.compile(r'\w+', _re.UNICODE)
def uni_toks(t): return _UNI.findall(t.lower())
def _lcs_py(a,b):
    if not a or not b: return 0
    dp=[0]*(len(b)+1)
    for ai in a:
        prev=0
        for j,bj in enumerate(b):
            cur=dp[j+1]; dp[j+1]=prev+1 if ai==bj else max(dp[j+1],dp[j]); prev=cur
    return dp[-1]
def lcs_tok(a,b):
    if HAVE_PYLCS:
        v={}
        for t in a:
            if t not in v: v[t]=len(v)
        for t in b:
            if t not in v: v[t]=len(v)
        return pylcs.lcs_sequence_length(''.join(chr(0x100+v[t]) for t in a), ''.join(chr(0x100+v[t]) for t in b))
    return _lcs_py(a,b)
def uni_r1(rt,ht):
    if not rt or not ht: return 0.0
    return 2*sum((Counter(rt)&Counter(ht)).values())/(len(rt)+len(ht))
def uni_rl(rt,ht):
    if not rt or not ht: return 0.0
    return 2*lcs_tok(rt,ht)/(len(rt)+len(ht))
def get_same_lang_candidates(q_text,q_emb,subset,k=K_CANDIDATES,exclude_exact=True):
    qs=q_text.strip()
    if subset in lang_indices:
        idx,mask=lang_indices[subset]
        D,I=idx.search(np.asarray(q_emb,np.float32).reshape(1,-1),min(k+5,len(mask)))
        out=[]
        for d,li in zip(D[0],I[0]):
            if li<0: continue
            ci=mask[int(li)]
            if exclude_exact and corpus_q_stripped[ci]==qs: continue
            out.append({'answer':answers_raw[ci],'sim':float(d),'idx':ci})
            if len(out)>=k: break
        if out: return out
    return []
def union4(q_text,afri_q,gem_q,bge_q,subset,exclude_exact=True):
    qs=q_text.strip(); rrf={}
    for idx_map,emb in [(lang_indices,afri_q),(gem_lang_idx,gem_q),(qa_idx,afri_q),(bge_idx,bge_q)]:
        if subset not in idx_map: continue
        idx,mask=idx_map[subset]
        D,I=idx.search(np.asarray(emb,np.float32).reshape(1,-1),min(K_LEG+5,len(mask)))
        r=0
        for li in I[0]:
            if li<0: continue
            ci=mask[int(li)]
            if exclude_exact and corpus_q_stripped[ci]==qs: continue
            rrf[ci]=rrf.get(ci,0.0)+1.0/(60+r); r+=1
            if r>=K_LEG: break
    ranked=sorted(rrf.items(),key=lambda kv:-kv[1])[:35]
    return [{'answer':answers_raw[ci],'sim':float(np.dot(afri_q,corpus_emb[ci])),'idx':ci} for ci,_ in ranked]
def uni_prep(cands,max_tok=80):
    answers=[c['answer'] for c in cands]
    w=np.exp(np.array([c['sim'] for c in cands])*5); w/=w.sum()
    seen,dd,ddw={},[],[]
    for a,wi in zip(answers,w):
        k=a.strip().lower()
        if k in seen: ddw[seen[k]]+=wi
        else: seen[k]=len(dd); dd.append(a); ddw.append(wi)
    ddw=np.array(ddw); ddw/=ddw.sum(); n=len(dd)
    toks=[uni_toks(a)[:max_tok] for a in dd]
    if n==1: return dd,ddw,np.zeros(1),np.zeros(1)
    S1,SL=np.zeros((n,n)),np.zeros((n,n))
    for i in range(n):
        for j in range(i+1,n):
            S1[i,j]=S1[j,i]=uni_r1(toks[i],toks[j]); SL[i,j]=SL[j,i]=uni_rl(toks[i],toks[j])
    return dd,ddw,S1@ddw,SL@ddw
def mbr_idx(ub,ddw,alpha,margin):
    if len(ddw)==1: return 0
    u=ub+alpha*ddw; b=int(np.argmax(u))
    return b if (b!=0 and u[b]-u[0]>margin) else 0
uni_prior={sub: float(np.median([len(uni_toks(answers_raw[i])) for i,s in enumerate(subsets_raw) if s==sub])) for sub in SUBSET_TO_LANG}
def uni_stitch(cands,lam,sub):
    answers=[c['answer'] for c in cands]
    w=np.exp(np.array([c['sim'] for c in cands])*5); w/=w.sum()
    p=Counter()
    for a,wi in zip(answers,w):
        for t,c in Counter(uni_toks(a)[:CAP]).items(): p[t]+=wi*c
    ref_len=max(lam*uni_prior[sub],1.0); seen,pool=set(),[]
    for a in answers:
        for s in _re.split(r'(?<=[.!?])\s+|\n+',a):
            s=s.strip(); st=uni_toks(s)
            if len(st)<4: continue
            k=' '.join(st)
            if k not in seen: seen.add(k); pool.append((s,Counter(st),len(st)))
    if not pool: return answers[0]
    def ef1(cov,hl):
        mt=sum(min(c,p[t]) for t,c in cov.items())
        if hl==0 or mt==0: return 0.0
        pr_,rc=mt/hl,mt/ref_len; return 2*pr_*rc/(pr_+rc)
    chosen,cov,hl,cur=[],Counter(),0,0.0
    while pool:
        bi,bg=-1,0.0
        for i,(s,sc_,sl) in enumerate(pool):
            nc=cov.copy(); nc.update(sc_)
            g=ef1(nc,hl+sl)-cur
            if g>bg: bg,bi=g,i
        if bi<0: break
        s,sc_,sl=pool.pop(bi); chosen.append(s); cov.update(sc_); hl+=sl; cur+=bg
    return ' '.join(chosen) if chosen else answers[0]

# ---- tuned decisions from the unicode rebuild (hardcoded = reproducible) ----
choice = {'Aka_Gha':('2leg',0.10,0.010),'Amh_Eth':('2leg',0.05,99.0),'Eng_Eth':('2leg',0.05,99.0),
          'Eng_Gha':('4leg',0.20,0.050),'Eng_Ken':('2leg',0.05,99.0),'Eng_Uga':('2leg',0.05,99.0),
          'Lug_Uga':('2leg',0.05,99.0),'Swa_Ken':('2leg',0.05,99.0)}
uni_stitch_gate = {'Aka_Gha':{'use':True,'lam':0.70,'pool':'2leg'},
                   'Eng_Gha':{'use':True,'lam':0.85,'pool':'4leg'},
                   'Amh_Eth':{'use':True,'lam':0.70,'pool':'2leg'}}
print("STATE RESTORED:", len(combined), "corpus |", len(llm_ans), "LLM answers | all caches loaded")

STATE RESTORED: 36501 corpus | 2618 LLM answers | all caches loaded


In [7]:
# ==== LB SIMULATOR v2: test-mix reweighting + weighted objective ====
test_mix = test_df['subset'].value_counts(normalize=True).to_dict()
val_mix  = val_df['subset'].value_counts(normalize=True).to_dict()
print(f"{'Sub':<10} {'val %':>7} {'test %':>7}   <- if these differ, our val was miscalibrated")
for sub in sorted(SUBSET_TO_LANG):
    print(f"{sub:<10} {100*val_mix.get(sub,0):>6.1f}% {100*test_mix.get(sub,0):>6.1f}%")

# Reweighted current-pipeline val score (unicode, per-language choices)
sim_r1 = sim_rl = 0.0
for sub in sorted(SUBSET_TO_LANG):
    tag, a, m = choice[sub]
    pr, rf = P[tag]['prep'], P[tag]['ref']
    idxs = [i for i in range(len(val_df)) if str(val_df.iloc[i]['subset'])==sub and rf[i]]
    g = uni_stitch_gate.get(sub, {})
    if g.get('use'):
        pools = val_cands_all if g['pool']=='2leg' else v4c
        r1 = np.mean([uni_r1(uni_toks(str(val_df.iloc[i]['output']))[:CAP],
              uni_toks(uni_stitch(pools[i], g['lam'], sub))[:CAP]) for i in idxs[1::2]])
    else:
        r1 = np.mean([rf[i][mbr_idx(pr[i][2], pr[i][1], a, m)][0] for i in idxs])
    rl = np.mean([rf[i][mbr_idx(pr[i][3], pr[i][1], a, m)][1] for i in idxs])
    w = test_mix.get(sub, 0)
    sim_r1 += w*r1; sim_rl += w*rl
print(f"\nTEST-WEIGHTED sim: R1={sim_r1:.4f} RL={sim_rl:.4f}")
print(f"LB actual:         R1=0.6632 RL=0.5875")
print(f"Sim weighted score (LLM@0.7854): {0.37*sim_r1+0.37*sim_rl+0.26*0.7854:.4f} vs LB 0.6670")

Sub          val %  test %   <- if these differ, our val was miscalibrated
Aka_Gha      16.7%   18.8%
Amh_Eth       6.9%    2.3%
Eng_Eth       8.4%    2.3%
Eng_Gha      16.5%   18.8%
Eng_Ken       5.8%    6.4%
Eng_Uga      25.2%   28.4%
Lug_Uga      12.7%   14.3%
Swa_Ken       7.7%    8.7%

TEST-WEIGHTED sim: R1=0.6677 RL=0.5981
LB actual:         R1=0.6632 RL=0.5875
Sim weighted score (LLM@0.7854): 0.6726 vs LB 0.6670


In [10]:
# ---- 2) FINE-TUNE (T4 memory-safe): 8-bit Adam + grad checkpointing ----
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
!pip install -q bitsandbytes
import bitsandbytes as bnb
import torch, gc

# clear whatever the failed attempt left on the GPU
try: model.cpu(); del model
except NameError: pass
gc.collect(); torch.cuda.empty_cache()
print(f"GPU free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

model = SentenceTransformer(str(AFRIE5_DIR) if AFRIE5_DIR.exists()
        else 'McGill-NLP/AfriE5-Large-instruct', device='cuda:0')
model.max_seq_length = 96
model[0].auto_model.gradient_checkpointing_enable()   # trade compute for memory

examples = [InputExample(texts=list(t)) for t in triplets]
loader = DataLoader(examples, shuffle=True, batch_size=8, drop_last=True)
loss = losses.MultipleNegativesRankingLoss(model)
model.fit(train_objectives=[(loader, loss)], epochs=1,
          warmup_steps=int(0.1 * len(loader)),
          optimizer_class=bnb.optim.AdamW8bit,         # 4x smaller optimizer states
          optimizer_params={'lr': 1e-5},
          use_amp=True, show_progress_bar=True)
model.save(str(FT2_DIR)); print(f"Saved: {FT2_DIR}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.0 MB/s eta 0:00:00
GPU free: 8.7 GB


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
500,0.795782


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Saved: /content/drive/MyDrive/multilingual-health-qa/outputs/afrie5-ft2


In [11]:
gc.collect(); torch.cuda.empty_cache()
print(f"GPU free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

GPU free: 7.6 GB


In [12]:
# ---- 3) RE-ENCODE with FT2 (model still in memory from training) ----
enc = lambda T: model.encode([f"{PREFIX}{t}" for t in T], batch_size=64,
        show_progress_bar=True, normalize_embeddings=True).astype(np.float32)
ft2_corpus = enc(questions_raw); np.save(CACHE / 'ft2_corpus.npy', ft2_corpus)
ft2_val    = enc(val_qs);        np.save(CACHE / 'ft2_val.npy', ft2_val)
ft2_test   = enc(test_qs);       np.save(CACHE / 'ft2_test.npy', ft2_test)   # encode test NOW too
model.cpu(); del model; gc.collect(); torch.cuda.empty_cache()
print("FT2 embeddings cached (corpus, val, test).")

Batches:   0%|          | 0/571 [00:00<?, ?it/s]

Batches:   0%|          | 0/105 [00:00<?, ?it/s]

Batches:   0%|          | 0/41 [00:00<?, ?it/s]

FT2 embeddings cached (corpus, val, test).


In [13]:
# ---- 4) SIMULATOR VERDICT: top-1 + oracle, old vs FT2, test-weighted ----
ft2_corpus = np.load(CACHE / 'ft2_corpus.npy')
ft2_val    = np.load(CACHE / 'ft2_val.npy')
ft2_idx = build_lang_idx(ft2_corpus)
test_mix = test_df['subset'].value_counts(normalize=True).to_dict()

print(f"{'Sub':<10} {'w%':>5} {'old T1R1':>9} {'FT2 T1R1':>9} {'Δ':>8} {'old orac':>9} {'FT2 orac':>9}")
old_w = new_w = 0.0
for sub in sorted(SUBSET_TO_LANG):
    idxs = [i for i in range(len(val_df)) if str(val_df.iloc[i]['subset']) == sub
            and str(val_df.iloc[i]['output']).strip()]
    o_t1, n_t1, o_orc, n_orc = [], [], [], []
    ix_o, mask_o = lang_indices[sub]; ix_n, mask_n = ft2_idx[sub]
    for i in idxs:
        ref = uni_toks(str(val_df.iloc[i]['output']))[:CAP]
        q = val_qs[i].strip()
        def top_pool(ix, mask, emb):
            D, I = ix.search(np.asarray(emb, np.float32).reshape(1, -1), 20)
            out = []
            for li in I[0]:
                if li < 0: continue
                ci = mask[int(li)]
                if corpus_q_stripped[ci] == q: continue
                out.append(uni_r1(ref, uni_toks(answers_raw[ci])[:CAP]))
                if len(out) >= 15: break
            return out
        po = top_pool(ix_o, mask_o, val_emb[i])
        pn = top_pool(ix_n, mask_n, ft2_val[i])
        if po: o_t1.append(po[0]); o_orc.append(max(po))
        if pn: n_t1.append(pn[0]); n_orc.append(max(pn))
    w = test_mix.get(sub, 0)
    old_w += w * np.mean(o_t1); new_w += w * np.mean(n_t1)
    print(f"{sub:<10} {100*w:>4.1f}% {np.mean(o_t1):>9.4f} {np.mean(n_t1):>9.4f} "
          f"{np.mean(n_t1)-np.mean(o_t1):>+8.4f} {np.mean(o_orc):>9.4f} {np.mean(n_orc):>9.4f}")
print(f"\nTEST-WEIGHTED top-1 R1:  old={old_w:.4f}  FT2={new_w:.4f}  Δ={new_w-old_w:+.4f}")
print("ADOPT if Δ ≥ +0.005 and no strong language regressed >0.003.")

Sub           w%  old T1R1  FT2 T1R1        Δ  old orac  FT2 orac
Aka_Gha    18.8%    0.3307    0.3441  +0.0134    0.4407    0.4423
Amh_Eth     2.3%    0.2076    0.2059  -0.0018    0.3301    0.3302
Eng_Eth     2.3%    0.6955    0.6877  -0.0078    0.7782    0.7786
Eng_Gha    18.8%    0.3331    0.3498  +0.0167    0.4296    0.4428
Eng_Ken     6.4%    0.8991    0.8828  -0.0163    0.9654    0.9645
Eng_Uga    28.4%    0.8707    0.8342  -0.0365    0.9361    0.9455
Lug_Uga    14.3%    0.8256    0.7854  -0.0401    0.9177    0.9043
Swa_Ken     8.7%    0.9484    0.9366  -0.0118    0.9827    0.9819

TEST-WEIGHTED top-1 R1:  old=0.6511  FT2=0.6383  Δ=-0.0128
ADOPT if Δ ≥ +0.005 and no strong language regressed >0.003.


In [15]:
def uni_refsc(ref, dd):
    rt = uni_toks(ref)[:CAP]
    return [(uni_r1(rt, uni_toks(c)[:CAP]), uni_rl(rt, uni_toks(c)[:CAP])) for c in dd]

In [16]:
# =============================================================================
# FT2-GHA LEG: add FT2 retrieval to Aka_Gha/Eng_Gha pools only -> sim verdict
# =============================================================================
ft2_corpus = np.load(CACHE/'ft2_corpus.npy'); ft2_val = np.load(CACHE/'ft2_val.npy')
ft2_test = np.load(CACHE/'ft2_test.npy')
ft2_idx = build_lang_idx(ft2_corpus)
GHA = ['Aka_Gha', 'Eng_Gha']

def union5_gha(q_text, afri_q, gem_q, bge_q, ft2_q, subset, exclude_exact=True):
    qs = q_text.strip(); rrf = {}
    legs = [(lang_indices, afri_q), (gem_lang_idx, gem_q), (qa_idx, afri_q),
            (bge_idx, bge_q), (ft2_idx, ft2_q)]
    for idx_map, emb in legs:
        if subset not in idx_map: continue
        idx, mask = idx_map[subset]
        D, I = idx.search(np.asarray(emb, np.float32).reshape(1, -1), min(K_LEG+5, len(mask)))
        r = 0
        for li in I[0]:
            if li < 0: continue
            ci = mask[int(li)]
            if exclude_exact and corpus_q_stripped[ci] == qs: continue
            rrf[ci] = rrf.get(ci, 0.0) + 1.0/(60+r); r += 1
            if r >= K_LEG: break
    ranked = sorted(rrf.items(), key=lambda kv: -kv[1])[:35]
    return [{'answer': answers_raw[ci], 'sim': float(np.dot(afri_q, corpus_emb[ci])), 'idx': ci}
            for ci, _ in ranked]

print(f"{'Sub':<10} {'cur best R1':>11} {'5leg MBR':>9} {'5leg+stitch':>11} {'cur orac':>9} {'5leg orac':>9}")
gha_adopt = {}
for sub in GHA:
    idxs = [i for i in range(len(val_df)) if str(val_df.iloc[i]['subset']) == sub
            and str(val_df.iloc[i]['output']).strip()]
    pools5 = {i: union5_gha(val_qs[i].strip(), val_emb[i], gem_val[i], bge_val[i],
              ft2_val[i], sub) for i in tqdm(idxs, desc=f"5leg {sub}", leave=False)}
    tune, hold = idxs[0::2], idxs[1::2]
    # current best on holdout (existing choice incl. stitch)
    tag = uni_stitch_gate[sub]['pool']; lam = uni_stitch_gate[sub]['lam']
    pools_cur = val_cands_all if tag == '2leg' else v4c
    cur_r1 = np.mean([uni_r1(uni_toks(str(val_df.iloc[i]['output']))[:CAP],
             uni_toks(uni_stitch(pools_cur[i], lam, sub))[:CAP]) for i in hold])
    cur_orc = np.mean([max(uni_r1(uni_toks(str(val_df.iloc[i]['output']))[:CAP],
              uni_toks(c['answer'])[:CAP]) for c in pools_cur[i]) for i in hold if pools_cur[i]])
    # 5-leg: tune (alpha,margin) + lambda on tune half, evaluate hold
    prep5 = {i: uni_prep(pools5[i]) for i in idxs if pools5[i]}
    ref5 = {i: uni_refsc(str(val_df.iloc[i]['output']).strip(), prep5[i][0]) for i in prep5}
    def m5(ix, a, m):
        return np.mean([ref5[i][mbr_idx(prep5[i][2], prep5[i][1], a, m)][0] for i in ix if i in prep5])
    a5, mm5 = max([(a, m) for a in [0.05,0.1,0.15,0.2] for m in [0.0,0.005,0.01,0.02,0.03,0.05]],
                  key=lambda p: m5(tune, *p))
    mbr5 = m5(hold, a5, mm5)
    def st5(ix, lam_):
        return np.mean([uni_r1(uni_toks(str(val_df.iloc[i]['output']))[:CAP],
               uni_toks(uni_stitch(pools5[i], lam_, sub))[:CAP]) for i in ix if pools5[i]])
    lam5 = max([0.7, 0.85, 1.0], key=lambda l: st5(tune, l))
    sti5 = st5(hold, lam5)
    orc5 = np.mean([max(r[0] for r in ref5[i]) for i in hold if i in ref5])
    best5 = max(mbr5, sti5)
    gha_adopt[sub] = {'adopt': best5 > cur_r1 + 0.005, 'use_stitch': sti5 >= mbr5,
                      'a': a5, 'm': mm5, 'lam': lam5}
    print(f"{sub:<10} {cur_r1:>11.4f} {mbr5:>9.4f} {sti5:>11.4f} {cur_orc:>9.4f} {orc5:>9.4f}")
print("\nDecisions:", gha_adopt)
print("Adopt per language only if 5leg best > current+0.005 on holdout.")

Sub        cur best R1  5leg MBR 5leg+stitch  cur orac 5leg orac


Aka_Gha         0.3839    0.3636      0.3906    0.4381    0.4565


Eng_Gha         0.3676    0.3546      0.3700    0.4502    0.4520

Decisions: {'Aka_Gha': {'adopt': np.True_, 'use_stitch': np.True_, 'a': 0.2, 'm': 0.03, 'lam': 0.7}, 'Eng_Gha': {'adopt': np.False_, 'use_stitch': np.True_, 'a': 0.2, 'm': 0.05, 'lam': 1.0}}
Adopt per language only if 5leg best > current+0.005 on holdout.


In [1]:
# =============================================================================
# EXPERIMENT B: Qwen2.5-7B-Instruct grounded refinement -> rubric-judged on val
# Open-source replacement for the Gemini LLM column. T4, 4-bit.
# =============================================================================
!pip install -q transformers accelerate bitsandbytes
import torch, gc, json, re as _re2
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

gc.collect(); torch.cuda.empty_cache()
MID = 'Qwen/Qwen2.5-7B-Instruct'
tok = AutoTokenizer.from_pretrained(MID)
llm = AutoModelForCausalLM.from_pretrained(MID, device_map='cuda:0',
    quantization_config=BitsAndBytesConfig(load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type='nf4'))
llm.eval(); print("Qwen2.5-7B loaded (4-bit)")

def qwen_refine(q, lang, cands, max_new=320):
    ctx = "\n".join(f"{k+1}. {c['answer']}" for k, c in enumerate(cands[:4]))
    msgs = [{"role": "system", "content":
             f"You are a professional health expert answering in {lang}. "
             f"Base your answer strictly on the reference answers given. Merge complementary "
             f"points. Be complete, factually accurate, and direct. Reply ONLY with the answer, "
             f"in {lang}."},
            {"role": "user", "content": f"Question: {q}\n\nReference answers:\n{ctx}"}]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt').to('cuda:0')
    with torch.no_grad():
        out = llm.generate(ids, max_new_tokens=max_new, do_sample=False,
                           temperature=None, top_p=None, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()

# ---- generate on the SAME 400-query val sample used for Gemini (direct A/B) ----
rng = np.random.default_rng(42)
val_sample = []
for sub in SUBSET_TO_LANG:
    idxs = [i for i in range(len(val_df)) if str(val_df.iloc[i]['subset']) == sub and val_cands_all[i]]
    val_sample += [int(x) for x in rng.choice(idxs, min(50, len(idxs)), replace=False)]

qprog = OUTPUT_DIR / 'qwen_val_sample.json'
qans = json.load(open(qprog)) if qprog.exists() else {}
for n, i in enumerate(tqdm(val_sample, desc="Qwen val")):
    if str(i) in qans: continue
    qans[str(i)] = qwen_refine(val_qs[i], SUBSET_TO_LANG[str(val_df.iloc[i]['subset'])],
                               val_cands_all[i])
    if (n+1) % 25 == 0: json.dump(qans, open(qprog, 'w'))
json.dump(qans, open(qprog, 'w')); print(f"Qwen val sample: {len(qans)}")

# ---- judge with the PUBLISHED rubric (1-5: accuracy, completeness, language) ----
# Qwen judges all three options blind; relative comparison is what we need.
def rubric_judge(q, ref, cand):
    msgs = [{"role": "system", "content":
             "You grade answers to health questions. Score the candidate 1-5 considering: "
             "factual accuracy vs the reference, completeness, and language appropriateness "
             "(same language as the question). Reply with ONLY the integer."},
            {"role": "user", "content":
             f"Question: {q}\n\nReference answer:\n{ref}\n\nCandidate answer:\n{cand}\n\nScore:"}]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt').to('cuda:0')
    with torch.no_grad():
        out = llm.generate(ids, max_new_tokens=4, do_sample=False, pad_token_id=tok.eos_token_id)
    m = _re2.search(r'[1-5]', tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True))
    return int(m.group()) if m else None

gem_vans = json.load(open(OUTPUT_DIR / 'gemini_val_sample.json'))
print(f"\n{'Sub':<10} {'n':>4} {'top1':>6} {'gemini':>7} {'qwen':>6}")
for sub in sorted(SUBSET_TO_LANG):
    rows = []
    for i in [x for x in val_sample if str(val_df.iloc[x]['subset']) == sub][:50]:
        ref = str(val_df.iloc[i]['output']).strip()
        if not ref: continue
        s = {}
        for name, cand in (('top1', val_cands_all[i][0]['answer']),
                           ('gem', gem_vans.get(str(i), '')),
                           ('qwen', qans[str(i)])):
            if cand: s[name] = rubric_judge(val_qs[i], ref, cand)
        if len(s) == 3 and all(v is not None for v in s.values()): rows.append(s)
    if rows:
        print(f"{sub:<10} {len(rows):>4} {np.mean([r['top1'] for r in rows]):>6.2f} "
              f"{np.mean([r['gem'] for r in rows]):>7.2f} {np.mean([r['qwen'] for r in rows]):>6.2f}")
print("\nRead RELATIVELY: if qwen ≈ gemini per language, the open-source swap is free.")
print("If qwen << gemini on African languages (likely Luganda/Akan/Amharic), gate per language:")
print("qwen where it holds, top-1 retrieval where it doesn't.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Qwen2.5-7B loaded (4-bit)


NameError: name 'np' is not defined